### 1. CDD model

#### Calculate the optimal model parameters

In [ ]:
import os
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyswarm import pso
import importlib
import pyswarm
importlib.reload(pyswarm)
from joblib import Parallel, delayed
from tqdm import tqdm


def plot_pso_convergence(global_best_fitness, row, col):
    """Plot PSO convergence curve"""
    plt.figure(figsize=(10, 6))
    plt.plot(global_best_fitness, marker='o', color='blue', label="PSO Convergence")
    plt.title(f"PSO Convergence Trend (Row: {row}, Col: {col})")
    plt.xlabel("Iteration")
    plt.ylabel("Global Best Fitness")
    plt.grid()
    plt.legend()
    plt.show()

def mad_based_nan(points, threshold=2.5):
    """Identify outliers using MAD and replace them with NaN"""
    points = points.copy()  # Remove read-only restriction
    median = np.median(points, axis=0)  # Calculate median
    diff = np.abs(points - median)  # Calculate absolute deviation from median
    mad = np.median(diff)  # Calculate MAD
    outlier_mask = diff > (threshold * mad)  # Mark outliers exceeding threshold
    points[outlier_mask] = np.nan  # Replace outliers with NaN
    return points

def process_single_pixel(col_idx, phenology_row_data, temp_row_data): # Process single pixel
    try:
        phenology_pixel = phenology_row_data[col_idx, :] # Get all years' data for column col
        temp_pixel = temp_row_data[col_idx, :, :] # Get all days' data for all years
        phenology_pixel = phenology_pixel.squeeze()
        temp_pixel = temp_pixel.squeeze()

        if np.isnan(temp_pixel).all():   
            return [np.nan, np.nan, np.nan]

        # Data cleaning - remove nan values
        temp_copy = np.full_like(temp_pixel, np.nan, dtype=np.float32)
        
        for year in range(22):  
            temp_year = temp_pixel[year, :]
            
            if np.isnan(temp_year).sum() > 0:
                temp_year = pd.Series(temp_year).interpolate(
                    method='linear', limit_direction='both'
                ).fillna(method='ffill').fillna(method='bfill').to_numpy()
            temp_copy[year, :] = temp_year

        # Remove outliers
        phenology_pixel = mad_based_nan(phenology_pixel)
        
        if ((phenology_pixel < 180).sum() + np.isnan(phenology_pixel).sum()) > 0.7 * len(phenology_pixel):   # If invalid values exceed 30% of total data
            return [np.nan, np.nan, np.nan]

        # Fill missing values
        if np.isnan(phenology_pixel).sum() > 0:
            phenology_pixel = pd.Series(phenology_pixel).interpolate(
                method='linear', limit_direction='both'
            ).fillna(method='ffill').fillna(method='bfill').to_numpy()

        def objective_function(para, phenology_data, temp_data):
            """
            Python implementation of SM.phenology function.
            
            Parameters:
                x: Parameter list [Ps, Tb, x, y, Ycrit]
                temp_data: Temperature data, shape (years, days)
                hours_data: Phenology data, shape days (same for each year)
                phenology_data: Observed phenology data
            
            Returns:
                RMSE: Calculated root mean square error
            """
            sim = []
            
            Tb = para[0]   # Temperature threshold
            Ycrit = para[1] # Accumulation threshold
            
            # Calculate estimated DFS for each year
            for year in range(len(phenology_data)):
                temp01 = temp_data[year,:]
                Cacc = 0
                
                Cacc = Tb - temp01
                Cacc = np.where((temp01 >= Tb) | (np.arange(len(temp01)) < 183), 0, Cacc)
                Sf = np.cumsum(Cacc)
                DFS = np.argmax(Sf >= Ycrit) + 1  # +1 to convert from index to day
                if Sf.max() < Ycrit:
                    sim.append(365)
                else:
                    sim.append(DFS)

            sim = np.array(sim)
            rmse = np.sqrt(np.mean((sim - phenology_data) ** 2))
            
            return rmse

        lb = [0, 0]  # Lower bounds
        ub = [50, 30000]  # Upper bounds

        xopt, rmse, fg_histor = pso(
            objective_function,
            lb,
            ub,
            args=(phenology_pixel, temp_copy),
            swarmsize=50,   # Number of particles
            maxiter=100     # Maximum iterations
        )

        Tb, Ycrit = xopt

        return [Tb, Ycrit, rmse]  # Return 3 parameters
    except Exception as e:
        print(f"Error processing column {col_idx}: {e}")
        return [np.nan, np.nan, np.nan]

output_folder = r'H:\DFS_models\result\CDD\CDD_para'
os.makedirs(output_folder, exist_ok=True)

# Data input
DFS_data = np.load(r'H:\DFS_models\DFS.npy')
DFS_data = DFS_data[:,1:23]
print(DFS_data.shape)

temp_data = np.load(r'H:\DFS_models\temp.npy')
print(temp_data.shape)

indices = np.arange(452)

results = Parallel(n_jobs=-1)(
    delayed(process_single_pixel)(
        idx, DFS_data, temp_data
    )
    for idx in tqdm(indices))

results = np.array(results)
print(results.shape)

output_file = os.path.join(output_folder, f"CDD_para.npz")
np.savez(output_file, results)

#### Calculate model performance

In [ ]:
import os
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyswarm import pso
import importlib
import pyswarm
importlib.reload(pyswarm)
from joblib import Parallel, delayed
from tqdm import tqdm
import netCDF4 as nc
import warnings
from scipy.stats import pearsonr


def KGE(obs, pre):
    """
    Calculate Kling-Gupta Efficiency (KGE)
    
    Parameters:
        obs: ndarray, observed values
        pre: ndarray, predicted values
        
    Returns:
        kge_value: float, KGE value
    """
    # Calculate Pearson correlation coefficient
    r, _ = pearsonr(obs, pre)
    
    # Calculate mean and standard deviation
    m_obs = np.mean(obs)  # Mean of observed values
    m_pre = np.mean(pre)  # Mean of predicted values
    std_obs = np.std(obs)  # Standard deviation of observed values
    std_pre = np.std(pre)  # Standard deviation of predicted values
    
    # Calculate Kling-Gupta Efficiency
    kge_value = 1 - np.sqrt((r - 1)**2 + (std_pre / std_obs - 1)**2 + (m_pre / m_obs - 1)**2)
    
    return kge_value


def calculate_AIC(observed, simulated, k):
    """
    Calculate AIC (Akaike Information Criterion)
    Reference: That RSE paper about nighttime lights
    
    Parameters:
        observed: ndarray, observed values
        simulated: ndarray, simulated values
        k: int, number of parameters
        
    Returns:
        aic: float, AIC value
    """
    n = len(observed)
    residual = observed - simulated
    mse = np.mean(residual**2)  # Mean squared error
    aic = n * np.log(mse) + 2 * k
    return aic


# Only pixels with average phenology values will be calculated (mask)
def cal_phenology(para_pixel, phenology_pixel, temp_copy):
    """
    Python implementation of SM.phenology function.
    
    Parameters:
        para_pixel: parameter array [Ps, Tb, x, y, Ycrit, rmse]
        temp_data: temperature data, shape (years, days)
        phenology_data: observed phenology data
        
    Returns:
        rmse, corr, p_value, kge, aic_value
    """
    sim = []
    
    Tb = para_pixel[0]   # Temperature threshold
    Ycrit = para_pixel[1] # Accumulation threshold
    
    # Calculate estimated DFS for each year
    for year in range(22):
        temp01 = temp_copy[year,:]

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            # Run code that might generate warnings
            Cacc = Tb - temp01
    
        Cacc = np.where((temp01 >= Tb) | (np.arange(len(temp01)) < 183), 0, Cacc)
        Sf = np.cumsum(Cacc)
        DFS = np.argmax(Sf >= Ycrit) + 1  # +1 to convert from index to day
        if Sf.max() < Ycrit:
            sim.append(365)
        else:
            sim.append(DFS)

    sim = np.array(sim)   # Ensure sim values aren't all identical
    
    # Calculate RMSE
    rmse = np.sqrt(np.mean((sim - phenology_pixel) ** 2))

    # Calculate Pearson correlation coefficient R and p-value
    corr, p_value = pearsonr(sim, phenology_pixel)

    # Calculate KGE
    kge = KGE(phenology_pixel, sim)

    # Calculate AIC
    k = 2 # Number of model parameters
    aic_value = calculate_AIC(phenology_pixel, sim, k)

    return rmse, corr, p_value, kge, aic_value


# Function to calculate day length
def daylength(day, latitude):
    """
    Calculate day length (in hours) for a specific latitude and day of year (DOY).
    
    Parameters:
        day (int or ndarray): Day of year (DOY)
        latitude (float or ndarray): Latitude, positive for northern hemisphere
        
    Returns:
        hours (float or ndarray): Day length in hours
    """
    # CBM Model parameters
    theta = 0.2163108 + 2 * np.arctan(0.9671396 * np.tan(0.00860 * (day - 186)))  # Solar annual angle
    declination = np.arcsin(0.39795 * np.cos(theta))  # Solar declination angle
    
    # Sun elevation angle: angle between sun's vertex and horizon (includes refraction correction)
    p = 0.8333  # Sun elevation angle at sunrise/sunset (degrees)
    
    # Calculate day length
    inter01 = (np.sin(np.radians(p)) + 
               np.sin(np.radians(latitude)) * np.sin(declination)) / (
               np.cos(np.radians(latitude)) * np.cos(declination))
    
    # Limit value range [-1, 1]
    inter01 = np.clip(inter01, -1, 1)
    
    # Calculate day length
    hours = 24 - (24 / np.pi) * np.arccos(inter01)
    return hours


def mad_based_nan(points, threshold=2.5):
    """ Identify outliers using MAD and replace them with NaN """
    median = np.median(points, axis=0)  # Calculate median
    diff = np.abs(points - median)  # Calculate absolute deviation from median
    mad = np.median(diff)  # Calculate MAD
    outlier_mask = diff > (threshold * mad)  # Mark outliers exceeding threshold
    points[outlier_mask] = np.nan  # Replace outliers with NaN
    return points


def process_single_pixel(col_idx, para, phenology_row_data, temp_row_data): # Process single pixel
    phenology_pixel = phenology_row_data[col_idx, :] # Get all years' data for column col
    temp_pixel = temp_row_data[col_idx, :, :] # Get all days' data for all years
    para_pixel = para[col_idx, :]

    phenology_pixel = phenology_pixel.squeeze()
    temp_pixel = temp_pixel.squeeze()
    para_pixel = para_pixel.squeeze()

    # Data cleaning - remove nan values
    temp_copy = np.full_like(temp_pixel, np.nan, dtype=np.float32)
    
    for year in range(22):  # Corresponding to 2003-2020
        temp_year = temp_pixel[year, :]
        
        if np.isnan(temp_year).sum() > 0:
            temp_year = pd.Series(temp_year).interpolate(
                method='linear', limit_direction='both'
            ).fillna(method='ffill').fillna(method='bfill').to_numpy()
        temp_copy[year, :] = temp_year

    # Remove outliers
    phenology_pixel = mad_based_nan(phenology_pixel)
    
    # Fill missing values
    if np.isnan(phenology_pixel).sum() > 0:
        phenology_pixel = pd.Series(phenology_pixel).interpolate(
            method='linear', limit_direction='both'
        ).fillna(method='ffill').fillna(method='bfill').to_numpy()

    sim_DFS = cal_phenology(para_pixel, phenology_pixel, temp_copy)

    return sim_DFS


output_folder = r'H:\DFS_models\result\CDD\CDD_performance'
os.makedirs(output_folder, exist_ok=True)

# Data input
DFS_data = np.load(r'H:\DFS_models\DFS.npy')
DFS_data = DFS_data[:,1:23]
print(DFS_data.shape)

temp_data = np.load(r'H:\DFS_models\temp.npy')
print(temp_data.shape)

para_data = np.load(r"H:\DFS_models\result\CDD\CDD_para\CDD_para.npz")
para_data = para_data['arr_0']
print(para_data.shape)

indices = np.arange(452)

results = Parallel(n_jobs=-1)(
    delayed(process_single_pixel)(  
        idx, para_data, DFS_data, temp_data
    )
    for idx in indices
)
results = np.array(results)
print(results.shape)

np.save(os.path.join(output_folder, 'CDD_performance.npy'), results)

### 2. CDD_ALAN model

#### Calculate the optimal model parameters

In [ ]:
import os
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyswarm import pso
import importlib
import pyswarm
importlib.reload(pyswarm)
from joblib import Parallel, delayed
from tqdm import tqdm
import netCDF4 as nc
import warnings

def plot_pso_convergence(global_best_fitness, row, col):
    """Plot PSO convergence curve"""
    plt.figure(figsize=(10, 6))
    plt.plot(global_best_fitness, marker='o', color='blue', label="PSO Convergence")
    plt.title(f"PSO Convergence Trend (Row: {row}, Col: {col})")
    plt.xlabel("Iteration")
    plt.ylabel("Global Best Fitness")
    plt.grid()
    plt.legend()
    plt.show()

def mad_based_nan(points, threshold=2.5):
    """Identify outliers using MAD and replace them with NaN"""
    points = points.copy()  # Remove read-only restriction
    median = np.median(points, axis=0)  # Calculate median
    diff = np.abs(points - median)  # Calculate absolute deviation from median
    mad = np.median(diff)  # Calculate MAD
    outlier_mask = diff > (threshold * mad)  # Mark outliers exceeding threshold
    points[outlier_mask] = np.nan  # Replace outliers with NaN
    return points

def process_single_pixel(col_idx, phenology_row_data, temp_row_data, alan_row_data): # Process single pixel
    try:
        phenology_pixel = phenology_row_data[col_idx, :] # Get all years' data for column col
        temp_pixel = temp_row_data[col_idx, :, :] # Get all days' data for all years
        alan_pixel = alan_row_data[col_idx, :]
        phenology_pixel = phenology_pixel.squeeze()
        temp_pixel = temp_pixel.squeeze()
        alan_pixel = alan_pixel.squeeze()

        if np.isnan(temp_pixel).all():   # Skip if all temperature data is NaN
            return [np.nan, np.nan, np.nan, np.nan]

        # Data cleaning - remove NaN values
        temp_copy = np.full_like(temp_pixel, np.nan, dtype=np.float32)
        
        for year in range(22):  # Corresponding to 2001-2022
            temp_year = temp_pixel[year, :]
            
            if np.isnan(temp_year).sum() > 0:
                temp_year = pd.Series(temp_year).interpolate(
                    method='linear', limit_direction='both'
                ).fillna(method='ffill').fillna(method='bfill').to_numpy()
            temp_copy[year, :] = temp_year

        # Remove outliers
        phenology_pixel = mad_based_nan(phenology_pixel)
        
        if ((phenology_pixel < 180).sum() + np.isnan(phenology_pixel).sum()) > 0.7 * len(phenology_pixel):   # If invalid values exceed 30% of total data
            return [np.nan, np.nan, np.nan, np.nan]

        # Fill missing values
        if np.isnan(phenology_pixel).sum() > 0:
            phenology_pixel = pd.Series(phenology_pixel).interpolate(
                method='linear', limit_direction='both'
            ).fillna(method='ffill').fillna(method='bfill').to_numpy()

        def objective_function(para, phenology_data, temp_data, alan_data):
            """
            Python implementation of SM.phenology function with ALAN impact.
            
            Parameters:
                para: parameter list [Tb, k, Ycrit]
                temp_data: temperature data, shape (years, days)
                phenology_data: phenology observation data
                alan_data: artificial light at night data
                
            Returns:
                RMSE: root mean square error
            """
            sim = []
            
            Tb = para[0]   # Temperature threshold
            k = para[1]    # ALAN impact coefficient
            Ycrit = para[2] # Accumulation threshold
            alan_mean = np.max(alan_data)  # Maximum ALAN value
            
            # Calculate estimated DFS for each year
            for year in range(len(phenology_data)):
                temp01 = temp_data[year,:]
                
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", category=RuntimeWarning)
                    # Run code that might generate warnings
                    Cacc = (Tb - temp01) * np.exp(k * ((alan_data[year] - alan_mean) / alan_mean))

                Cacc = np.where((temp01 >= Tb) | (np.arange(len(temp01)) < 183), 0, Cacc)
                Sf = np.cumsum(Cacc)
                DFS = np.argmax(Sf >= Ycrit) + 1  # +1 to convert from index to day
                if Sf.max() < Ycrit:
                    sim.append(365)
                else:
                    sim.append(DFS)

            sim = np.array(sim)
            rmse = np.sqrt(np.mean((sim - phenology_data) ** 2))
            
            return rmse

        # Define parameter bounds (reference: MATLAB program and Liu Qiang's GCB)
        lb = [0, -2, 0]      # Lower bounds [Tb, k, Ycrit]
        ub = [50, 2, 30000]  # Upper bounds (temperature max set to 50 for testing)

        # Run PSO optimization
        xopt, rmse, fg_histor = pso(
            objective_function,
            lb,
            ub,
            args=(phenology_pixel, temp_copy, alan_pixel),
            swarmsize=50,   # Number of particles
            maxiter=100     # Maximum iterations
        )

        Tb, k, Ycrit = xopt

        return Tb, k, Ycrit, rmse  # Return 4 parameters

    except Exception as e:
        print(f"Error processing column {col_idx}: {e}")
        return [np.nan, np.nan, np.nan, np.nan]

# Main program
output_folder = r'H:\DFS_models\result\CDD_ALAN\CDD_alan_para'
os.makedirs(output_folder, exist_ok=True)

# Data loading
DFS_data = np.load(r'H:\DFS_models\DFS.npy')
DFS_data = DFS_data[:,1:23]
print(DFS_data.shape)

temp_data = np.load(r'H:\DFS_models\temp.npy')
print(temp_data.shape)

ALAN_data = np.load(r'H:\DFS_models\ALAN.npy')
ALAN_data = ALAN_data[:,1:23]
print(ALAN_data.shape)

indices = np.arange(452)

# Parallel processing
results = Parallel(n_jobs=-1)(
    delayed(process_single_pixel)(
        idx, DFS_data, temp_data, ALAN_data
    )
    for idx in tqdm(indices))

results = np.array(results)
print(results.shape)

# Save results
output_file = os.path.join(output_folder, f"CDD_alan_para.npz")
np.savez(output_file, results)

#### Calculate model performance

In [ ]:
import os
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyswarm import pso
import importlib
import pyswarm
importlib.reload(pyswarm)
from joblib import Parallel, delayed
from tqdm import tqdm
import netCDF4 as nc
import warnings
from scipy.stats import pearsonr


def KGE(obs, pre):
    """
    Calculate Kling-Gupta Efficiency (KGE)
    
    Parameters:
        obs: ndarray, observed values
        pre: ndarray, predicted values
        
    Returns:
        kge_value: float, KGE value
    """
    # Calculate Pearson correlation coefficient
    r, _ = pearsonr(obs, pre)
    
    # Calculate mean and standard deviation
    m_obs = np.mean(obs)  # Mean of observed values
    m_pre = np.mean(pre)  # Mean of predicted values
    std_obs = np.std(obs)  # Standard deviation of observed values
    std_pre = np.std(pre)  # Standard deviation of predicted values
    
    # Calculate Kling-Gupta Efficiency
    kge_value = 1 - np.sqrt((r - 1)**2 + (std_pre / std_obs - 1)**2 + (m_pre / m_obs - 1)**2)
    
    return kge_value


def calculate_AIC(observed, simulated, k):
    """
    Calculate AIC (Akaike Information Criterion)
    Reference: That RSE paper about nighttime lights
    
    Parameters:
        observed: ndarray, observed values
        simulated: ndarray, simulated values
        k: int, number of parameters
        
    Returns:
        aic: float, AIC value
    """
    n = len(observed)
    residual = observed - simulated
    mse = np.mean(residual**2)  # Mean squared error
    aic = n * np.log(mse) + 2 * k
    return aic


# Only pixels with average phenology values will be calculated (mask)
def cal_phenology(para_pixel, phenology_pixel, temp_copy, ALAN_pixel):
    """
    Python implementation of SM.phenology function with ALAN impact.
    
    Parameters:
        para_pixel: parameter array [Tb, k, Ycrit]
        temp_data: temperature data, shape (years, days)
        phenology_data: observed phenology data
        ALAN_data: artificial light at night data
        
    Returns:
        rmse, corr, p_value, kge, aic_value
    """
    sim = []
    
    Tb = para_pixel[0]   # Temperature threshold
    k = para_pixel[1]    # ALAN impact coefficient
    Ycrit = para_pixel[2] # Accumulation threshold
    
    alan_mean = np.max(ALAN_pixel)  # Maximum ALAN value
    
    # Calculate estimated DFS for each year
    for year in range(22):
        temp01 = temp_copy[year,:]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            # Run code that might generate warnings
            Cacc = (Tb - temp01) * np.exp(k * ((ALAN_pixel[year] - alan_mean) / alan_mean))

        Cacc = np.where((temp01 >= Tb) | (np.arange(len(temp01)) < 183), 0, Cacc)
        Sf = np.cumsum(Cacc)
        DFS = np.argmax(Sf >= Ycrit) + 1  # +1 to convert from index to day
        if Sf.max() < Ycrit:
            sim.append(365)
        else:
            sim.append(DFS)

    sim = np.array(sim)

    # Calculate RMSE
    rmse = np.sqrt(np.mean((sim - phenology_pixel) ** 2))

    # Calculate Pearson correlation coefficient and p-value
    corr, p_value = pearsonr(sim, phenology_pixel)

    # Calculate KGE
    kge = KGE(phenology_pixel, sim)

    # Calculate AIC
    k = 3 # Number of model parameters
    aic_value = calculate_AIC(phenology_pixel, sim, k)

    return rmse, corr, p_value, kge, aic_value


def daylength(day, latitude):
    """
    Calculate day length (in hours) for a specific latitude and day of year (DOY).
    
    Parameters:
        day (int or ndarray): Day of year (DOY)
        latitude (float or ndarray): Latitude, positive for northern hemisphere
        
    Returns:
        hours (float or ndarray): Day length in hours
    """
    # CBM Model parameters
    theta = 0.2163108 + 2 * np.arctan(0.9671396 * np.tan(0.00860 * (day - 186)))  # Solar annual angle
    declination = np.arcsin(0.39795 * np.cos(theta))  # Solar declination angle
    
    # Sun elevation angle: angle between sun's vertex and horizon (includes refraction correction)
    p = 0.8333  # Sun elevation angle at sunrise/sunset (degrees)
    
    # Calculate day length
    inter01 = (np.sin(np.radians(p)) + 
               np.sin(np.radians(latitude)) * np.sin(declination)) / (
               np.cos(np.radians(latitude)) * np.cos(declination))
    
    # Limit value range [-1, 1]
    inter01 = np.clip(inter01, -1, 1)
    
    # Calculate day length
    hours = 24 - (24 / np.pi) * np.arccos(inter01)
    return hours


def mad_based_nan(points, threshold=2.5):
    """ Identify outliers using MAD and replace them with NaN """
    median = np.median(points, axis=0)  # Calculate median
    diff = np.abs(points - median)  # Calculate absolute deviation from median
    mad = np.median(diff)  # Calculate MAD
    outlier_mask = diff > (threshold * mad)  # Mark outliers exceeding threshold
    points[outlier_mask] = np.nan  # Replace outliers with NaN
    return points


def process_single_pixel(col_idx, para, phenology_row_data, temp_row_data, ALAN_row_data):
    """Process single pixel"""
    phenology_pixel = phenology_row_data[col_idx, :] # Get all years' data for column
    temp_pixel = temp_row_data[col_idx, :, :] # Get all days' data for all years
    para_pixel = para[col_idx, :]
    ALAN_pixel = ALAN_row_data[col_idx, :]
    ALAN_pixel = ALAN_pixel.squeeze()
    phenology_pixel = phenology_pixel.squeeze()
    temp_pixel = temp_pixel.squeeze()
    para_pixel = para_pixel.squeeze()

    # Data cleaning - remove NaN values
    temp_copy = np.full_like(temp_pixel, np.nan, dtype=np.float32)
    
    for year in range(22):  # Corresponding to 2003-2020
        temp_year = temp_pixel[year, :]
        
        if np.isnan(temp_year).sum() > 0:
            temp_year = pd.Series(temp_year).interpolate(
                method='linear', limit_direction='both'
            ).fillna(method='ffill').fillna(method='bfill').to_numpy()
        temp_copy[year, :] = temp_year

    # Remove outliers
    phenology_pixel = mad_based_nan(phenology_pixel)
    
    # Fill missing values
    if np.isnan(phenology_pixel).sum() > 0:
        phenology_pixel = pd.Series(phenology_pixel).interpolate(
            method='linear', limit_direction='both'
        ).fillna(method='ffill').fillna(method='bfill').to_numpy()

    sim_DFS = cal_phenology(para_pixel, phenology_pixel, temp_copy, ALAN_pixel)

    return sim_DFS


# Main program
output_folder = r'H:\DFS_models\result\CDD_ALAN\CDD_alan_performance'
os.makedirs(output_folder, exist_ok=True)

# Data loading
DFS_data = np.load(r'H:\DFS_models\DFS.npy')
DFS_data = DFS_data[:,1:23]
print(DFS_data.shape)

ALAN_data = np.load(r'H:\DFS_models\ALAN.npy')
ALAN_data = ALAN_data[:,1:23]
print(ALAN_data.shape)

temp_data = np.load(r'H:\DFS_models\temp.npy')
print(temp_data.shape)

para_data = np.load(r"H:\DFS_models\result\CDD_ALAN\CDD_alan_para\CDD_alan_para.npz")
para_data = para_data['arr_0']
print(para_data.shape)

indices = np.arange(452)

# Parallel processing
results = Parallel(n_jobs=-1)(
    delayed(process_single_pixel)(  
        idx, para_data, DFS_data, temp_data, ALAN_data
    )
    for idx in indices
)
results = np.array(results)
print(results.shape)

# Save results
np.save(os.path.join(output_folder, 'CDD_alan_performance.npy'), results)

In [ ]:
import numpy as np

# Load the original performance metrics array
CDD_alan_performance = np.load(r"H:\DFS_models\result\CDD_ALAN\CDD_alan_performance\CDD_alan_performance.npy")

# Verify array dimensions (should have 5 metrics in second dimension)
print("Array shape:", CDD_alan_performance.shape)

# Define metric names corresponding to each column
metric_names = ['rmse', 'corr', 'p_value', 'kge', 'aic_value']

# Split and save each metric as separate file
for idx, metric in enumerate(metric_names):
    output_path = fr"H:\DFS_models\result\CDD_ALAN\CDD_alan_performance\{metric}.npy"
    np.save(output_path, CDD_alan_performance[:, idx])
    print(f"Saved {metric} to {output_path}")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

# Input and output directories
input_dir = r"H:\DFS_models\result\CDD_ALAN\CDD_alan_performance"
output_dir = r"H:\DFS_models\result\CDD_ALAN\CDD_alan_performance\distribution"

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)

# Process all .npy files in the directory
for filename in os.listdir(input_dir):
    if filename.endswith(".npy"):
        file_path = os.path.join(input_dir, filename)
        print("Processing:", filename)   
        
        # Load .npy file
        data = np.load(file_path)
        
        # Remove NaN values
        valid_data = data[~np.isnan(data)]

        # Get base filename without extension
        base_filename = os.path.splitext(filename)[0]

        # Calculate density distribution
        density = gaussian_kde(valid_data)
        x_vals = np.linspace(valid_data.min(), valid_data.max(), 500)
        y_vals = density(x_vals)

        # Plot density distribution
        plt.figure(figsize=(8, 6))
        plt.plot(x_vals, y_vals, color='darkblue', lw=2, label="Density")
        plt.fill_between(x_vals, y_vals, color='skyblue', alpha=0.4)
        plt.xlabel(f"{base_filename}", fontsize=14)  # Add filename to x-axis label
        plt.ylabel("Density", fontsize=14)
        plt.legend(fontsize=12)
        plt.grid(alpha=0.3)
        
        # Save density plot
        density_output_path = os.path.join(output_dir, f"{base_filename}_density.png")
        plt.savefig(density_output_path)
        plt.close()

        print(f"Completed processing: {filename}")

print("All files processed successfully!")

### 3. DM model

#### Calculate the optimal model parameters

In [ ]:
import os
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyswarm import pso
import importlib
import pyswarm
importlib.reload(pyswarm)
from joblib import Parallel, delayed
from tqdm import tqdm
import netCDF4 as nc
import warnings

def plot_pso_convergence(global_best_fitness, row, col):
    """Plot PSO convergence curve"""
    plt.figure(figsize=(10, 6))
    plt.plot(global_best_fitness, marker='o', color='blue', label="PSO Convergence")
    plt.title(f"PSO Convergence Trend (Row: {row}, Col: {col})")
    plt.xlabel("Iteration")
    plt.ylabel("Global Best Fitness")
    plt.grid()
    plt.legend()
    plt.show()

def mad_based_nan(points, threshold=2.5):
    """Identify outliers using MAD and replace them with NaN"""
    points = points.copy()  # Remove read-only restriction
    median = np.median(points, axis=0)  # Calculate median
    diff = np.abs(points - median)  # Calculate absolute deviation from median
    mad = np.median(diff)  # Calculate MAD
    outlier_mask = diff > (threshold * mad)  # Mark outliers exceeding threshold
    points[outlier_mask] = np.nan  # Replace outliers with NaN
    return points

def process_single_pixel(col_idx, phenology_row_data, temp_row_data, hours_row_data):
    """Process single pixel"""
    try:
        phenology_pixel = phenology_row_data[col_idx, :]  # Get all years' data for column
        temp_pixel = temp_row_data[col_idx, :, :]  # Get all days' data for all years
        phenology_pixel = phenology_pixel.squeeze()
        temp_pixel = temp_pixel.squeeze()
        
        hours_pixel = hours_row_data[col_idx, :]
        hours_pixel = hours_pixel.squeeze()

        if np.isnan(temp_pixel).all():  # Skip if all temperature data is NaN
            return [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan]

        # Data cleaning - remove NaN values
        temp_copy = np.full_like(temp_pixel, np.nan, dtype=np.float32)
        
        for year in range(22):  # Corresponding to 2001-2022
            temp_year = temp_pixel[year, :]
            
            if np.isnan(temp_year).sum() > 0:
                temp_year = pd.Series(temp_year).interpolate(
                    method='linear', limit_direction='both'
                ).fillna(method='ffill').fillna(method='bfill').to_numpy()
            temp_copy[year, :] = temp_year

        # Remove outliers
        phenology_pixel = mad_based_nan(phenology_pixel)
        
        if ((phenology_pixel < 180).sum() + np.isnan(phenology_pixel).sum()) > 0.7 * len(phenology_pixel):
            return [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan]

        # Fill missing values
        if np.isnan(phenology_pixel).sum() > 0:
            phenology_pixel = pd.Series(phenology_pixel).interpolate(
                method='linear', limit_direction='both'
            ).fillna(method='ffill').fillna(method='bfill').to_numpy()

        def objective_function(para, phenology_data, temp_data, hours_data):
            """
            Python implementation of SM.phenology function
            
            Parameters:
                para: parameter list [Ps, Tb, x, y, Ycrit]
                temp_data: temperature data, shape (years, days)
                hours_data: photoperiod data, shape (days,)
                phenology_data: observed phenology data
                
            Returns:
                RMSE: root mean square error
            """
            sim = []
            
            Ps = para[0]  # Photoperiod threshold
            Tb = para[1]  # Temperature threshold
            x = para[2]   # Parameter x
            y = para[3]   # Parameter y
            Ycrit = para[4]  # Accumulation threshold
            
            # Calculate estimated DFS for each year
            for year in range(len(phenology_data)):
                temp01 = temp_data[year,:]
                
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", category=RuntimeWarning)
                    Cacc = (Tb - temp01) ** x * (1 - hours_data / Ps) ** y

                Cacc = np.where((temp01 >= Tb) | (hours_data >= Ps) | (np.arange(len(temp01)) < 183), 0, Cacc)
                Sf = np.cumsum(Cacc)
                DFS = np.argmax(Sf >= Ycrit) + 1  # +1 to convert from index to day
                if Sf.max() < Ycrit:
                    sim.append(365)
                else:
                    sim.append(DFS)

            sim = np.array(sim)
            rmse = np.sqrt(np.mean((sim - phenology_data) ** 2))
            
            return rmse

        # Define parameter bounds (reference: MATLAB program and Liu Qiang's GCB)
        lb = [8, 0, 0, 0, 0]      # Lower bounds [Ps, Tb, x, y, Ycrit]
        ub = [24, 50, 2, 2, 30000] # Upper bounds (temperature max set to 50 for testing)

        # Run PSO optimization
        xopt, rmse, fg_histor = pso(
            objective_function,
            lb,
            ub,
            args=(phenology_pixel, temp_copy, hours_pixel),
            swarmsize=50,   # Number of particles
            maxiter=100     # Maximum iterations
        )

        Ps, Tb, x, y, Ycrit = xopt

        return Ps, Tb, x, y, Ycrit, rmse  # Return 6 parameters
    
    except Exception as e:
        print(f"Error processing column {col_idx}: {e}")
        return [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan]

# Main program
output_folder = r'H:\DFS_models\result\DM\DM_para'
os.makedirs(output_folder, exist_ok=True)

# Load input data
DFS_data = np.load(r'H:\DFS_models\DFS.npy')
DFS_data = DFS_data[:,1:23]
print("DFS data shape:", DFS_data.shape)

hours_data = np.load(r'H:\DFS_models\photoperiod.npy')
hours_data = hours_data[:,1:366]
print("Photoperiod data shape:", hours_data.shape)

temp_data = np.load(r'H:\DFS_models\temp.npy')
print("Temperature data shape:", temp_data.shape)

# Process all pixels in parallel
indices = np.arange(452)
results = Parallel(n_jobs=-1)(
    delayed(process_single_pixel)(
        idx, DFS_data, temp_data, hours_data
    )
    for idx in tqdm(indices)
)

results = np.array(results)
print("Results shape:", results.shape)

# Save results
output_file = os.path.join(output_folder, "DM_para.npz")
np.savez(output_file, results)

#### Calculate model performance

In [ ]:

import os
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyswarm import pso
import importlib
import pyswarm
importlib.reload(pyswarm)
from joblib import Parallel, delayed
from tqdm import tqdm
import netCDF4 as nc
import warnings
from scipy.stats import pearsonr


def KGE(obs, pre):
    """
    Compute Kling-Gupta Efficiency (KGE)

    Args:
        obs: ndarray, observed values
        pre: ndarray, predicted values

    Returns:
        kge_value: float, KGE score
    """
    r, _ = pearsonr(obs, pre)
    m_obs = np.mean(obs)
    m_pre = np.mean(pre)
    std_obs = np.std(obs)
    std_pre = np.std(pre)

    kge_value = 1 - np.sqrt((r - 1) ** 2 + (std_pre / std_obs - 1) ** 2 + (m_pre / m_obs - 1) ** 2)
    return kge_value


def calculate_AIC(observed, simulated, k):
    """
    Compute Akaike Information Criterion (AIC)
    
    Args:
        observed: ndarray, observed values
        simulated: ndarray, predicted values
        k: int, number of model parameters

    Returns:
        aic: float, AIC score
    """
    n = len(observed)
    residual = observed - simulated
    mse = np.mean(residual ** 2)
    aic = n * np.log(mse) + 2 * k
    return aic


def cal_phenology(para_pixel, phenology_pixel, temp_copy, hours_pixel):
    """
    Phenology estimation function based on parameters.

    Args:
        para_pixel: parameter array [Ps, Tb, x, y, Ycrit, rmse]
        phenology_pixel: observed phenology
        temp_copy: temperature data (years, days)
        hours_pixel: photoperiod data

    Returns:
        rmse, correlation, p_value, KGE, AIC
    """
    sim = []
    Ps = para_pixel[0]
    Tb = para_pixel[1]
    x = para_pixel[2]
    y = para_pixel[3]
    Ycrit = para_pixel[4]

    for year in range(22):
        temp01 = temp_copy[year, :]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            Cacc = (Tb - temp01) ** x * (1 - hours_pixel / Ps) ** y

        Cacc = np.where((temp01 >= Tb) | (hours_pixel >= Ps) | (np.arange(len(temp01)) < 183), 0, Cacc)
        Sf = np.cumsum(Cacc)
        DFS = np.argmax(Sf >= Ycrit) + 1
        if Sf.max() < Ycrit:
            sim.append(365)
        else:
            sim.append(DFS)

    sim = np.array(sim)
    rmse = np.sqrt(np.mean((sim - phenology_pixel) ** 2))
    corr, p_value = pearsonr(sim, phenology_pixel)
    kge = KGE(phenology_pixel, sim)
    k = 5  # number of model parameters
    aic_value = calculate_AIC(phenology_pixel, sim, k)

    return rmse, corr, p_value, kge, aic_value


def daylength(day, latitude):
    """
    Calculate day length (hours) for a given day of year and latitude.

    Args:
        day: int or array-like, day of year (DOY)
        latitude: float or array-like, latitude (positive = Northern Hemisphere)

    Returns:
        hours: float or array, day length in hours
    """
    theta = 0.2163108 + 2 * np.arctan(0.9671396 * np.tan(0.00860 * (day - 186)))
    declination = np.arcsin(0.39795 * np.cos(theta))
    p = 0.8333  # solar elevation angle

    inter01 = (np.sin(np.radians(p)) + 
               np.sin(np.radians(latitude)) * np.sin(declination)) / (
               np.cos(np.radians(latitude)) * np.cos(declination))

    inter01 = np.clip(inter01, -1, 1)
    hours = 24 - (24 / np.pi) * np.arccos(inter01)
    return hours


def mad_based_nan(points, threshold=2.5):
    """Identify outliers using MAD and replace them with NaN"""
    median = np.median(points, axis=0)
    diff = np.abs(points - median)
    mad = np.median(diff)
    outlier_mask = diff > (threshold * mad)
    points[outlier_mask] = np.nan
    return points


def process_single_pixel(col_idx, para, phenology_row_data, temp_row_data, hours_row_data):
    """Process a single pixel for performance metrics"""
    phenology_pixel = phenology_row_data[col_idx, :]
    temp_pixel = temp_row_data[col_idx, :, :]
    para_pixel = para[col_idx, :]

    hours_pixel = hours_row_data[col_idx, :]
    hours_pixel = hours_pixel.squeeze()

    phenology_pixel = phenology_pixel.squeeze()
    temp_pixel = temp_pixel.squeeze()
    para_pixel = para_pixel.squeeze()

    temp_copy = np.full_like(temp_pixel, np.nan, dtype=np.float32)
    for year in range(22):  # years 2001–2022
        temp_year = temp_pixel[year, :]
        if np.isnan(temp_year).sum() > 0:
            temp_year = pd.Series(temp_year).interpolate(
                method='linear', limit_direction='both'
            ).fillna(method='ffill').fillna(method='bfill').to_numpy()
        temp_copy[year, :] = temp_year

    phenology_pixel = mad_based_nan(phenology_pixel)

    if np.isnan(phenology_pixel).sum() > 0:
        phenology_pixel = pd.Series(phenology_pixel).interpolate(
            method='linear', limit_direction='both'
        ).fillna(method='ffill').fillna(method='bfill').to_numpy()

    sim_DFS = cal_phenology(para_pixel, phenology_pixel, temp_copy, hours_pixel)

    return sim_DFS


# Output folder (must not contain Chinese characters)
output_folder = r'H:\DFS_models\result\DM\DM_performance'
os.makedirs(output_folder, exist_ok=True)

# Load data
DFS_data = np.load(r'H:\DFS_models\DFS.npy')
DFS_data = DFS_data[:, 1:23]
print(DFS_data.shape)

hours_data = np.load(r'H:\DFS_models\photoperiod.npy')
hours_data = hours_data[:, 1:366]
print(hours_data.shape)

temp_data = np.load(r'H:\DFS_models\temp.npy')
print(temp_data.shape)

para_data = np.load(r'H:\DFS_models\result\DM\DM_para\DM_para.npz')
para_data = para_data['arr_0']
print(para_data.shape)

indices = np.arange(452)

results = Parallel(n_jobs=-1)(
    delayed(process_single_pixel)(
        idx, para_data, DFS_data, temp_data, hours_data,
    )
    for idx in tqdm(indices)
)

results = np.array(results)
print(results.shape)

np.save(os.path.join(output_folder, 'DM_performance.npy'), results)

results = Parallel(n_jobs=-1)(
            delayed(process_single_pixel)(
                idx,para_data, DFS_data, temp_data, hours_data,

            )
            for idx in tqdm(indices))

results = np.array(results)
print(results.shape)



np.save(os.path.join(output_folder, 'DM_performance.npy'), results)

  


### 4. DM_ALAN model

#### Calculate the optimal model parameters

In [ ]:
import os
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyswarm import pso
import importlib
import pyswarm
importlib.reload(pyswarm)
from joblib import Parallel, delayed
from tqdm import tqdm
import netCDF4 as nc
import warnings

# Only pixels with valid mean phenology values are involved in the calculation (masking)

def plot_pso_convergence(global_best_fitness, row, col):
    """Plot PSO convergence curve"""
    plt.figure(figsize=(10, 6))
    plt.plot(global_best_fitness, marker='o', color='blue', label="PSO Convergence")
    plt.title(f"PSO Convergence Trend (Row: {row}, Col: {col})")
    plt.xlabel("Iteration")
    plt.ylabel("Global Best Fitness")
    plt.grid()
    plt.legend()
    plt.show()

def mad_based_nan(points, threshold=2.5):
    """ Detect outliers using MAD and replace them with NaN """
    points = points.copy()  # Remove read-only restriction
    median = np.median(points, axis=0)  # Compute median
    diff = np.abs(points - median)  # Absolute deviation from median
    mad = np.median(diff)  # Compute MAD
    outlier_mask = diff > (threshold * mad)  # Identify outliers
    points[outlier_mask] = np.nan  # Replace outliers with NaN
    return points

def process_single_pixel(col_idx, phenology_row_data, temp_row_data, hours_row_data, ALAN_row_data):
    """Process a single pixel"""
    try:
        phenology_pixel = phenology_row_data[col_idx, :]  # Get all yearly data for column idx
        temp_pixel = temp_row_data[col_idx, :, :]         # Get all daily data across years
        ALAN_pixel = ALAN_row_data[col_idx, :]            # Get ALAN data
        phenology_pixel = phenology_pixel.squeeze()
        temp_pixel = temp_pixel.squeeze()
        ALAN_pixel = ALAN_pixel.squeeze()
        
        hours_pixel = hours_row_data[col_idx, :]
        hours_pixel = hours_pixel.squeeze()

        # Data cleaning - remove NaNs
        temp_copy = np.full_like(temp_pixel, np.nan, dtype=np.float32)

        for year in range(22):  # Corresponds to 2001–2022
            temp_year = temp_pixel[year, :]
            if np.isnan(temp_year).sum() > 0:
                temp_year = pd.Series(temp_year).interpolate(
                    method='linear', limit_direction='both'
                ).fillna(method='ffill').fillna(method='bfill').to_numpy()
            temp_copy[year, :] = temp_year

        # Remove outliers
        phenology_pixel = mad_based_nan(phenology_pixel)

        # Fill missing values
        if np.isnan(phenology_pixel).sum() > 0:
            phenology_pixel = pd.Series(phenology_pixel).interpolate(
                method='linear', limit_direction='both'
            ).fillna(method='ffill').fillna(method='bfill').to_numpy()

        def objective_function(para, phenology_data, temp_data, hours_data, ALAN_data):
            """
            Python implementation of SM.phenology function.

            Parameters:
                para: Parameter list [Ps, Tb, x, y, k, Ycrit]
                temp_data: Temperature data (years x days)
                hours_data: Photoperiod data (1D, same for each year)
                phenology_data: Observed phenology (DFS)
            
            Returns:
                RMSE: Root Mean Square Error
            """
            sim = []

            Ps = para[0]    # Photoperiod threshold
            Tb = para[1]    # Temperature threshold
            x = para[2]     # Exponent x
            y = para[3]     # Exponent y
            k = para[4]     # ALAN factor
            Ycrit = para[5] # Accumulated threshold

            ALAN_mean = np.max(ALAN_data)

            for year in range(len(phenology_data)):
                temp01 = temp_data[year, :]
                Cacc = 0

                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", category=RuntimeWarning)
                    Cacc = (Tb - temp01) ** x * (1 - hours_data / Ps) ** y * np.exp(k * ((ALAN_data[year] - ALAN_mean) / ALAN_mean))

                Cacc = np.where((temp01 >= Tb) | (hours_data >= Ps) | (np.arange(len(temp01)) < 183), 0, Cacc)
                Sf = np.cumsum(Cacc)
                DFS = np.argmax(Sf >= Ycrit) + 1  # Convert index to day

                if Sf.max() < Ycrit:
                    sim.append(365)
                else:
                    sim.append(DFS)

            sim = np.array(sim)
            rmse = np.sqrt(np.mean((sim - phenology_data) ** 2))
            return rmse

        # Define bounds for parameters [Ps, Tb, x, y, k, Ycrit]
        # Based on MATLAB and literature (Liu Qiang, GCB)
        lb = [8, 0, 0, 0, -2, 0]
        ub = [24, 50, 2, 2, 2, 30000]

        # xopt is the optimized parameter set, rmse is the corresponding RMSE
        xopt, rmse, fg_histor = pso(
            objective_function,
            lb,
            ub,
            args=(phenology_pixel, temp_copy, hours_pixel, ALAN_pixel),
            swarmsize=50,
            maxiter=100
        )

        Ps, Tb, x, y, k, Ycrit = xopt

        return Ps, Tb, x, y, k, Ycrit, rmse  # Return 7 values

    except Exception as e:
        print(f"Error processing column {col_idx}: {e}")
        return [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan]

# Main program
# Initialize output folder
output_folder = r'H:\DFS_models\result\DM_ALAN\DM_alan_para'
os.makedirs(output_folder, exist_ok=True)

# Load input data
DFS_data = np.load(r'H:\DFS_models\DFS.npy')
DFS_data = DFS_data[:, 1:23]  # Use years 2001–2022
print(DFS_data.shape)

hours_data = np.load(r'H:\DFS_models\photoperiod.npy')
hours_data = hours_data[:, 1:366]  # Days 1–365
print(hours_data.shape)

temp_data = np.load(r'H:\DFS_models\temp.npy')
print(temp_data.shape)

ALAN_data = np.load(r'H:\DFS_models\ALAN.npy')
ALAN_data = ALAN_data[:, 1:23]  # Years 2001–2022
print(ALAN_data.shape)

# Process all 452 pixels
indices = np.arange(452)

results = Parallel(n_jobs=-1)(
    delayed(process_single_pixel)(
        idx, DFS_data, temp_data, hours_data, ALAN_data
    )
    for idx in tqdm(indices)
)

results = np.array(results)
print(results.shape)

# Save results
np.save(os.path.join(output_folder, 'DM_alan_para.npy'), results)
print("Saved successfully")


#### Calculate model performance

In [ ]:
import os
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyswarm import pso
import importlib
import pyswarm
importlib.reload(pyswarm)
from joblib import Parallel, delayed
from tqdm import tqdm
import netCDF4 as nc
import warnings
from scipy.stats import pearsonr


def KGE(obs, pre):
    """
    Calculate Kling-Gupta Efficiency (KGE)

    Parameters:
        obs: ndarray, observed values
        pre: ndarray, predicted values

    Returns:
        kge_value: float, KGE value
    """
    r, _ = pearsonr(obs, pre)

    m_obs = np.mean(obs)
    m_pre = np.mean(pre)
    std_obs = np.std(obs)
    std_pre = np.std(pre)

    kge_value = 1 - np.sqrt((r - 1)**2 + (std_pre / std_obs - 1)**2 + (m_pre / m_obs - 1)**2)

    return kge_value


def calculate_AIC(observed, simulated, k):
    """
    Calculate AIC (Akaike Information Criterion)
    Refer to the RSE paper on nighttime light

    Parameters:
        observed: ndarray, observed values
        simulated: ndarray, predicted values
        k: int, number of parameters

    Returns:
        aic: float, AIC value
    """
    n = len(observed)
    residual = observed - simulated
    mse = np.mean(residual**2)
    aic = n * np.log(mse) + 2 * k
    return aic


def cal_phenology(para_pixel, phenology_pixel, temp_copy, hours_pixel, ALAN_pixel):
    """
    Python implementation of SM.phenology function.

    Parameters:
        para_pixel: list, parameters for current pixel [Ps, Tb, x, y, k, Ycrit]
        phenology_pixel: observed phenology data
        temp_copy: temperature data, shape (years, days)
        hours_pixel: photoperiod data, same each year
        ALAN_pixel: ALAN values for each year

    Returns:
        rmse, corr, p_value, kge, aic_value: model performance metrics
    """
    sim = []

    Ps = para_pixel[0]
    Tb = para_pixel[1]
    x = para_pixel[2]
    y = para_pixel[3]
    k = para_pixel[4]
    Ycrit = para_pixel[5]

    alan_mean = np.max(ALAN_pixel)

    for year in range(22):
        temp01 = temp_copy[year, :]
        Cacc = 0

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            Cacc = (Tb - temp01) ** x * (1 - hours_pixel / Ps) ** y * np.exp(k * ((ALAN_pixel[year] - alan_mean) / alan_mean))

        Cacc = np.where((temp01 >= Tb) | (hours_pixel >= Ps) | (np.arange(len(temp01)) < 183), 0, Cacc)
        Sf = np.cumsum(Cacc)
        DFS = np.argmax(Sf >= Ycrit) + 1
        if Sf.max() < Ycrit:
            sim.append(365)
        else:
            sim.append(DFS)

    sim = np.array(sim)

    rmse = np.sqrt(np.mean((sim - phenology_pixel) ** 2))
    corr, p_value = pearsonr(sim, phenology_pixel)
    kge = KGE(phenology_pixel, sim)
    k = 5
    aic_value = calculate_AIC(phenology_pixel, sim, k)

    return rmse, corr, p_value, kge, aic_value


def daylength(day, latitude):
    """
    Calculate day length (hours) for a given DOY and latitude.

    Parameters:
        day (int or ndarray): day of year (DOY)
        latitude (float or ndarray): latitude in degrees

    Returns:
        hours (float or ndarray): day length in hours
    """
    theta = 0.2163108 + 2 * np.arctan(0.9671396 * np.tan(0.00860 * (day - 186)))
    declination = np.arcsin(0.39795 * np.cos(theta))

    p = 0.8333

    inter01 = (np.sin(np.radians(p)) +
               np.sin(np.radians(latitude)) * np.sin(declination)) / (
               np.cos(np.radians(latitude)) * np.cos(declination))

    inter01 = np.clip(inter01, -1, 1)
    hours = 24 - (24 / np.pi) * np.arccos(inter01)
    return hours


def mad_based_nan(points, threshold=2.5):
    """ Detect outliers using MAD and replace them with NaN """
    median = np.median(points, axis=0)
    diff = np.abs(points - median)
    mad = np.median(diff)
    outlier_mask = diff > (threshold * mad)
    points[outlier_mask] = np.nan
    return points


def process_single_pixel(col_idx, para, phenology_row_data, temp_row_data, hours_row_data, alan_row_data):
    """
    Process a single pixel.

    Parameters:
        col_idx: int, index of pixel
        para: parameter array
        phenology_row_data: phenology data
        temp_row_data: temperature data
        hours_row_data: photoperiod data
        alan_row_data: ALAN data

    Returns:
        Simulation metrics for the pixel
    """
    phenology_pixel = phenology_row_data[col_idx, :]
    temp_pixel = temp_row_data[col_idx, :, :]
    para_pixel = para[col_idx, :]

    hours_pixel = hours_row_data[col_idx, :].squeeze()
    alan_pixel = alan_row_data[col_idx, :].squeeze()
    phenology_pixel = phenology_pixel.squeeze()
    temp_pixel = temp_pixel.squeeze()
    para_pixel = para_pixel.squeeze()

    temp_copy = np.full_like(temp_pixel, np.nan, dtype=np.float32)

    for year in range(22):  # Corresponds to 2003–2020
        temp_year = temp_pixel[year, :]
        if np.isnan(temp_year).sum() > 0:
            temp_year = pd.Series(temp_year).interpolate(
                method='linear', limit_direction='both'
            ).fillna(method='ffill').fillna(method='bfill').to_numpy()
        temp_copy[year, :] = temp_year

    phenology_pixel = mad_based_nan(phenology_pixel)

    if np.isnan(phenology_pixel).sum() > 0:
        phenology_pixel = pd.Series(phenology_pixel).interpolate(
            method='linear', limit_direction='both'
        ).fillna(method='ffill').fillna(method='bfill').to_numpy()

    sim_DFS = cal_phenology(para_pixel, phenology_pixel, temp_copy, hours_pixel, alan_pixel)

    return sim_DFS


output_folder = r'H:\DFS_models\result\DM_ALAN\DM_alan_performance'
os.makedirs(output_folder, exist_ok=True)

# Load data
DFS_data = np.load(r'H:\DFS_models\DFS.npy')
DFS_data = DFS_data[:, 1:23]
print(DFS_data.shape)

ALAN_data = np.load(r'H:\DFS_models\ALAN.npy')
ALAN_data = ALAN_data[:, 1:23]
print(ALAN_data.shape)

hours_data = np.load(r'H:\DFS_models\photoperiod.npy')
hours_data = hours_data[:, 1:366]
print(hours_data.shape)

temp_data = np.load(r'H:\DFS_models\temp.npy')
print(temp_data.shape)

para_data = np.load(r'H:\DFS_models\result\DM_ALAN\DM_alan_para\DM_alan_para.npy')
print(para_data.shape)

indices = np.arange(452)

results = Parallel(n_jobs=-1)(
    delayed(process_single_pixel)(
        idx, para_data, DFS_data, temp_data, hours_data, ALAN_data
    )
    for idx in tqdm(indices)
)

results = np.array(results)
print(results.shape)

np.save(os.path.join(output_folder, 'DM_alan_performance.npy'), results)


In [ ]:
import numpy as np

# Load the performance metrics array
performance_data = np.load(r"H:\DFS_models\result\DM_ALAN\DM_alan_performance\DM_alan_performance.npy")

# Verify array dimensions
print("Array shape:", performance_data.shape)

# Define metric names corresponding to each column
metric_names = ['rmse', 'corr', 'p_value', 'kge', 'aic_value']

# Save each metric as a separate .npy file
for idx, metric in enumerate(metric_names):
    output_path = fr"H:\DFS_models\result\DM_ALAN\DM_alan_performance\{metric}.npy"
    np.save(output_path, performance_data[:, idx])
    print(f"Saved {metric} to {output_path}")

print("All metrics saved successfully!")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

# Set input and output directories
input_dir = r"H:\DFS_models\result\DM_ALAN\DM_alan_performance"
output_dir = r"H:\DFS_models\result\DM_ALAN\DM_alan_performance\distribution"

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Process all .npy files in the directory
for filename in os.listdir(input_dir):
    if filename.endswith(".npy"):
        file_path = os.path.join(input_dir, filename)
        print(f"Processing: {filename}")   
        
        # Load data file
        data = np.load(file_path)
        
        # Remove NaN values
        valid_data = data[~np.isnan(data)]

        # Get base filename without extension
        base_filename = os.path.splitext(filename)[0]

        # Calculate density distribution
        density = gaussian_kde(valid_data)
        x_vals = np.linspace(valid_data.min(), valid_data.max(), 500)
        y_vals = density(x_vals)

        # Plot density distribution
        plt.figure(figsize=(8, 6))
        plt.plot(x_vals, y_vals, color='darkblue', linewidth=2, label="Density")
        plt.fill_between(x_vals, y_vals, color='skyblue', alpha=0.4)
        plt.xlabel(f"{base_filename}", fontsize=14)  # Use filename as x-label
        plt.ylabel("Density", fontsize=14)
        plt.legend(fontsize=12)
        plt.grid(alpha=0.3)
        
        # Save density plot
        density_output_path = os.path.join(output_dir, f"{base_filename}_density.png")
        plt.savefig(density_output_path, dpi=300, bbox_inches='tight')
        plt.close()

        print(f"Completed processing: {filename}")

print("All files processed successfully!")

### 5. SIAM model

#### Calculate the optimal model parameters

In [ ]:
import os
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyswarm import pso
import importlib
import pyswarm
importlib.reload(pyswarm)
from joblib import Parallel, delayed
from tqdm import tqdm
import netCDF4 as nc
import warnings

def plot_pso_convergence(global_best_fitness, row, col):
    """Plot the PSO convergence curve"""
    plt.figure(figsize=(10, 6))
    plt.plot(global_best_fitness, marker='o', color='blue', label="PSO Convergence")
    plt.title(f"PSO Convergence Trend (Row: {row}, Col: {col})")
    plt.xlabel("Iteration")
    plt.ylabel("Global Best Fitness")
    plt.grid()
    plt.legend()
    plt.show()

def mad_based_nan(points, threshold=2.5):
    """Use MAD to detect outliers and replace them with NaN"""
    points = points.copy()  # Remove read-only restriction
    median = np.median(points, axis=0)
    diff = np.abs(points - median)
    mad = np.median(diff)
    outlier_mask = diff > (threshold * mad)
    points[outlier_mask] = np.nan
    return points

def process_single_pixel(col_idx, phenology_row_data, temp_row_data, hours_row_data, SOS_row_data):
    """Process a single pixel (column index)"""
    try:
        phenology_pixel = phenology_row_data[col_idx, :]
        SOS_pixel = SOS_row_data[col_idx, :]
        temp_pixel = temp_row_data[col_idx, :, :]
        phenology_pixel = phenology_pixel.squeeze()
        SOS_pixel = SOS_pixel.squeeze()
        temp_pixel = temp_pixel.squeeze()
        
        hours_pixel = hours_row_data[col_idx, :]
        hours_pixel = hours_pixel.squeeze()

        if np.isnan(temp_pixel).all():
            return [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan]

        # Fill missing temperature values
        temp_copy = np.full_like(temp_pixel, np.nan, dtype=np.float32)
        for year in range(22):  # Years 2001–2022
            temp_year = temp_pixel[year, :]
            if np.isnan(temp_year).sum() > 0:
                temp_year = pd.Series(temp_year).interpolate(
                    method='linear', limit_direction='both'
                ).fillna(method='ffill').fillna(method='bfill').to_numpy()
            temp_copy[year, :] = temp_year

        # Remove outliers in phenology
        phenology_pixel = mad_based_nan(phenology_pixel)

        if ((phenology_pixel < 180).sum() + np.isnan(phenology_pixel).sum()) > 0.7 * len(phenology_pixel):
            # If more than 70% of the data is invalid, skip
            return [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan]

        # Fill missing values in phenology
        if np.isnan(phenology_pixel).sum() > 0:
            phenology_pixel = pd.Series(phenology_pixel).interpolate(
                method='linear', limit_direction='both'
            ).fillna(method='ffill').fillna(method='bfill').to_numpy()

        # Define objective function
        def objective_function(para, phenology_data, temp_data, hours_data, SOS_data):
            """
            Simulate DFS using the phenology model.

            Parameters:
                para: parameter list [Ps, Tb, x, y, a, b]
                temp_data: temperature array (years, days)
                hours_data: photoperiod array (days)
                phenology_data: observed phenology values

            Returns:
                RMSE: root mean square error
            """
            sim = []
            Ps, Tb, x, y, a, b = para
            SOS_mean = np.mean(SOS_data)

            for year in range(len(phenology_data)):
                temp01 = temp_data[year, :]
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", category=RuntimeWarning)
                    Cacc = (Tb - temp01) ** x * (1 - hours_data / Ps) ** y

                Cacc = np.where((temp01 >= Tb) | (hours_data >= Ps) | (np.arange(len(temp01)) < 183), 0, Cacc)
                Sf = np.cumsum(Cacc)
                Ycrit = a + b * ((SOS_data[year] - SOS_mean) / SOS_mean)
                DFS = np.argmax(Sf >= Ycrit) + 1
                sim.append(365 if Sf.max() < Ycrit else DFS)

            sim = np.array(sim)
            rmse = np.sqrt(np.mean((sim - phenology_data) ** 2))
            return rmse

        def constraint(para, phenology_data, temp_data, hours_data, SOS_data):
            a, b = para[4], para[5]
            SOS_mean = np.mean(SOS_data)
            return a + b * ((SOS_data[year] - SOS_mean) / SOS_mean)

        # Parameter bounds (based on previous work)
        lb = [8, 0, 0, 0, 0, -1000]  # Lower bounds: Ps, Tb, x, y, a, b
        ub = [24, 50, 2, 2, 30000, 1000]  # Upper bounds

        xopt, rmse, fg_histor = pso(
            objective_function,
            lb,
            ub,
            ieqcons=[constraint],
            args=(phenology_pixel, temp_copy, hours_pixel, SOS_pixel),
            swarmsize=50,
            maxiter=100
        )

        Ps, Tb, x, y, a, b = xopt
        return Ps, Tb, x, y, a, b, rmse
    except Exception as e:
        return [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan]

# Main program
output_folder = r'H:\DFS_models\result\SIAM\SIAM_para'
os.makedirs(output_folder, exist_ok=True)

# Load data
DFS_data = np.load(r'H:\DFS_models\DFS.npy')
DFS_data = DFS_data[:, 1:23]
print(DFS_data.shape)

SOS_data = np.load(r'H:\DFS_models\SOS.npy')
SOS_data = SOS_data[:, 1:23]
print(SOS_data.shape)

hours_data = np.load(r'H:\DFS_models\photoperiod.npy')
hours_data = hours_data[:, 1:366]
print(hours_data.shape)

temp_data = np.load(r'H:\DFS_models\temp.npy')
print(temp_data.shape)

# Indices of pixels to process
indices = np.arange(452)

# Process each pixel in parallel
results = Parallel(n_jobs=-1)(
    delayed(process_single_pixel)(
        idx, DFS_data, temp_data, hours_data, SOS_data
    ) for idx in tqdm(indices)
)

results = np.array(results)
print(results.shape)

# Save result
np.save(os.path.join(output_folder, 'SIAM_para.npy'), results)


#### Calculate model performance

In [ ]:
import os
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyswarm import pso
import importlib
import pyswarm
importlib.reload(pyswarm)
from joblib import Parallel, delayed
from tqdm import tqdm
import netCDF4 as nc
import warnings
from scipy.stats import pearsonr

def KGE(obs, pre):
    """
    Calculate Kling-Gupta Efficiency (KGE)
    
    Parameters:
        obs: ndarray, observed values
        pre: ndarray, predicted values
        
    Returns:
        kge_value: float, KGE value
    """
    r, _ = pearsonr(obs, pre)  # Pearson correlation coefficient
    
    m_obs = np.mean(obs)
    m_pre = np.mean(pre)
    std_obs = np.std(obs)
    std_pre = np.std(pre)
    
    kge_value = 1 - np.sqrt((r - 1)**2 + (std_pre / std_obs - 1)**2 + (m_pre / m_obs - 1)**2)
    
    return kge_value

def calculate_AIC(observed, simulated, k):
    """
    Calculate AIC (Akaike Information Criterion)
    Reference: RSE paper on nighttime light
    
    Parameters:
        observed: ndarray, observed values
        simulated: ndarray, predicted values
        k: int, number of model parameters
        
    Returns:
        aic: float, AIC value
    """
    n = len(observed)
    residual = observed - simulated
    mse = np.mean(residual**2)
    aic = n * np.log(mse) + 2 * k
    return aic

# Only pixels with valid mean phenology are included (mask)
def cal_phenology(para_pixel, phenology_pixel, temp_copy, hours_pixel, SOS_pixel):
    """
    Python version of SM.phenology function
    
    Parameters:
        para_pixel: list of parameters [Ps, Tb, x, y, a, b]
        phenology_pixel: observed phenology
        temp_copy: temperature data (years × days)
        hours_pixel: photoperiod data (same each year)
        SOS_pixel: start of season data
        
    Returns:
        rmse, correlation, p_value, kge, aic
    """
    sim = []

    Ps = para_pixel[0]  # Photoperiod threshold
    Tb = para_pixel[1]  # Temperature threshold
    x = para_pixel[2]
    y = para_pixel[3]
    a = para_pixel[4]
    b = para_pixel[5]
    
    SOS_mean = np.mean(SOS_pixel)
    
    for year in range(22):
        temp01 = temp_copy[year, :]
        Cacc = 0

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            Cacc = (Tb - temp01) ** x * (1 - hours_pixel / Ps) ** y

        Cacc = np.where((temp01 >= Tb) | (hours_pixel >= Ps) | (np.arange(len(temp01)) < 183), 0, Cacc)
        Sf = np.cumsum(Cacc)
        Ycrit = a + b * ((SOS_pixel[year] - SOS_mean) / SOS_mean)
        DFS = np.argmax(Sf >= Ycrit) + 1
        sim.append(365 if Sf.max() < Ycrit else DFS)

    sim = np.array(sim)

    # Calculate RMSE
    rmse = np.sqrt(np.mean((sim - phenology_pixel) ** 2))

    # Calculate Pearson correlation and p-value
    corr, p_value = pearsonr(sim, phenology_pixel)

    # Calculate KGE
    kge = KGE(phenology_pixel, sim)

    # Calculate AIC
    k = 6  # Number of parameters
    aic_value = calculate_AIC(phenology_pixel, sim, k)

    return rmse, corr, p_value, kge, aic_value

def mad_based_nan(points, threshold=2.5):
    """ Detect outliers using MAD and replace them with NaN """
    points = np.copy(points)
    median = np.median(points, axis=0)
    diff = np.abs(points - median)
    mad = np.median(diff)
    outlier_mask = diff > (threshold * mad)
    points[outlier_mask] = np.nan
    return points

def process_single_pixel(col_idx, para_row_data, phenology_row_data, temp_row_data, hours_row_data, SOS_row_data):
    """ Process a single pixel """

    phenology_pixel = phenology_row_data[col_idx, :]
    SOS_pixel = SOS_row_data[col_idx, :]
    para_pixel = para_row_data[col_idx, :]
    temp_pixel = temp_row_data[col_idx, :, :]
    hours_pixel = hours_row_data[col_idx, :]

    phenology_pixel = phenology_pixel.squeeze()
    temp_pixel = temp_pixel.squeeze()
    para_pixel = para_pixel.squeeze()
    SOS_pixel = SOS_pixel.squeeze()
    hours_pixel = hours_pixel.squeeze()

    if np.isnan(temp_pixel).all():
        return [np.nan, np.nan, np.nan, np.nan, np.nan]

    temp_copy = np.full_like(temp_pixel, np.nan, dtype=np.float32)
    for year in range(22):  # 2003–2020
        temp_year = temp_pixel[year, :]
        if np.isnan(temp_year).sum() > 0:
            temp_year = pd.Series(temp_year).interpolate(method='linear', limit_direction='both') \
                .fillna(method='ffill').fillna(method='bfill').to_numpy()
        temp_copy[year, :] = temp_year

    # Remove outliers
    phenology_pixel = mad_based_nan(phenology_pixel)
    if ((phenology_pixel < 180).sum() + np.isnan(phenology_pixel).sum()) > 0.7 * len(phenology_pixel):
        return [np.nan, np.nan, np.nan, np.nan, np.nan]

    # Fill missing values
    if np.isnan(phenology_pixel).sum() > 0:
        phenology_pixel = pd.Series(phenology_pixel).interpolate(method='linear', limit_direction='both') \
            .fillna(method='ffill').fillna(method='bfill').to_numpy()

    sim_DFS = cal_phenology(para_pixel, phenology_pixel, temp_copy, hours_pixel, SOS_pixel)
    return sim_DFS

# Output folder
output_folder = r'H:\DFS_models\result\SIAM\SIAM_performance'
os.makedirs(output_folder, exist_ok=True)

# Load data
DFS_data = np.load(r'H:\DFS_models\DFS.npy')[:, 1:23]
print(DFS_data.shape)

SOS_data = np.load(r'H:\DFS_models\SOS.npy')[:, 1:23]
print(SOS_data.shape)

hours_data = np.load(r'H:\DFS_models\photoperiod.npy')[:, 1:366]
print(hours_data.shape)

temp_data = np.load(r'H:\DFS_models\temp.npy')
print(temp_data.shape)

para_data = np.load(r'H:\DFS_models\result\SIAM\SIAM_para\SIAM_para.npy')
print(para_data.shape)

# Indices to process
indices = np.arange(452)

# Parallel processing
results = Parallel(n_jobs=-1)(
    delayed(process_single_pixel)(idx, para_data, DFS_data, temp_data, hours_data, SOS_data)
    for idx in tqdm(indices)
)

results = np.array(results)
print(results.shape)
np.save(os.path.join(output_folder, 'SIAM_performance.npy'), results)


In [ ]:
import numpy as np

# Load the original array
CDD_performance = np.load(r"H:\DFS_models\result\SIAM\SIAM_performance\SIAM_performance.npy")

print("Shape:", CDD_performance.shape)


names = ['rmse', 'corr', 'p_value', 'kge', 'aic_value']
for i, name in enumerate(names):
    np.save(fr"H:\DFS_models\result\SIAM\SIAM_performance\{name}.npy", CDD_performance[:, i])


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

# Input and output directories
input_dir = r"H:\DFS_models\result\SIAM\SIAM_performance"
output_dir = r"H:\DFS_models\result\SIAM\SIAM_performance\distribution"

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)

# Iterate over all .npy files in the directory
for filename in os.listdir(input_dir):
    if filename.endswith(".npy"):
        file_path = os.path.join(input_dir, filename)
        print("Processing:", filename)   
        
        # Load .npy file
        data = np.load(file_path)
        
        # Remove NaN values
        valid_data = data[~np.isnan(data)]

        # Get the base filename without extension
        base_filename = os.path.splitext(filename)[0]

        # Calculate density distribution
        density = gaussian_kde(valid_data)
        x_vals = np.linspace(valid_data.min(), valid_data.max(), 500)
        y_vals = density(x_vals)

        # Plot density distribution
        plt.figure(figsize=(8, 6))
        plt.plot(x_vals, y_vals, color='darkblue', lw=2, label="Density")
        plt.fill_between(x_vals, y_vals, color='skyblue', alpha=0.4)
        plt.xlabel(f"{base_filename}", fontsize=14)  # Add filename as x-axis label
        plt.ylabel("Density", fontsize=14)
        plt.legend(fontsize=12)
        plt.grid(alpha=0.3)
        
        # Save density plot
        density_output_path = os.path.join(output_dir, f"{base_filename}_density.png")
        plt.savefig(density_output_path)
        plt.close()

        print(f"Completed processing: {filename}")

print("All files have been processed!")


### 6. SIAM_ALAN model

#### Calculate the optimal model parameters

In [ ]:
import os
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyswarm import pso
import importlib
import pyswarm
importlib.reload(pyswarm)
from joblib import Parallel, delayed
from tqdm import tqdm
import netCDF4 as nc
import warnings


def plot_pso_convergence(global_best_fitness, row, col):
    """Plot PSO convergence curve"""
    plt.figure(figsize=(10, 6))
    plt.plot(global_best_fitness, marker='o', color='blue', label="PSO Convergence")
    plt.title(f"PSO Convergence Trend (Row: {row}, Col: {col})")
    plt.xlabel("Iteration")
    plt.ylabel("Global Best Fitness")
    plt.grid()
    plt.legend()
    plt.show()


def mad_based_nan(points, threshold=2.5):
    """Use MAD to identify outliers and replace them with NaN"""
    points = points.copy()  # Remove read-only restriction
    median = np.median(points, axis=0)  # Calculate median
    diff = np.abs(points - median)  # Calculate absolute deviation from median
    mad = np.median(diff)  # Calculate MAD
    outlier_mask = diff > (threshold * mad)  # Mark outliers beyond threshold
    points[outlier_mask] = np.nan  # Replace outliers with NaN
    return points


def process_single_pixel(col_idx, phenology_row_data, temp_row_data, hours_row_data, SOS_row_data, alan_row_data):
    # Process a single pixel
    try:
        phenology_pixel = phenology_row_data[col_idx, :]  # Get data for column 'col_idx' across all years
        SOS_pixel = SOS_row_data[col_idx, :]              # Get SOS data for column 'col_idx' across all years
        alan_pixel = alan_row_data[col_idx, :]            # Get ALAN data for column 'col_idx' across all years
        temp_pixel = temp_row_data[col_idx, :, :]         # Get temperature data for all years and days
        phenology_pixel = phenology_pixel.squeeze()
        SOS_pixel = SOS_pixel.squeeze()
        temp_pixel = temp_pixel.squeeze()
        # print(temp_pixel.shape) #(22,365)
        
        hours_pixel = hours_row_data[col_idx, :]
        hours_pixel = hours_pixel.squeeze()

        if np.isnan(temp_pixel).all():
            return [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan]

        # Data cleaning - remove NaNs
        temp_copy = np.full_like(temp_pixel, np.nan, dtype=np.float32)

        for year in range(22):  # Corresponds to 2001-2022
            temp_year = temp_pixel[year, :]
            
            if np.isnan(temp_year).sum() > 0:
                temp_year = pd.Series(temp_year).interpolate(
                    method='linear', limit_direction='both'
                ).fillna(method='ffill').fillna(method='bfill').to_numpy()
            temp_copy[year, :] = temp_year

        # Remove outliers
        phenology_pixel = mad_based_nan(phenology_pixel)
        
        # If values less than 180 or NaNs exceed 70% of data, skip
        if ((phenology_pixel < 180).sum() + np.isnan(phenology_pixel).sum()) > 0.7 * len(phenology_pixel):
            return [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan]

        # Fill missing values
        if np.isnan(phenology_pixel).sum() > 0:
            phenology_pixel = pd.Series(phenology_pixel).interpolate(
                method='linear', limit_direction='both'
            ).fillna(method='ffill').fillna(method='bfill').to_numpy()


        def objective_function(para, phenology_data, temp_data, hours_data, SOS_data, alan_data):
            """
            Python implementation of SM.phenology function.
            
            Parameters:
                para: parameter list [Ps, Tb, x, y, a, b, k]
                temp_data: temperature data, shape (years, days)
                hours_data: photoperiod data, shape (days,), same for each year
                phenology_data: observed phenology data
            
            Returns:
                RMSE: calculated root mean square error
            """
            sim = []

            Ps = para[0]  # Photoperiod
            Tb = para[1]  # Temperature threshold
            x = para[2]   # parameter x
            y = para[3]   # parameter y
            a = para[4]   # cumulative threshold coefficient a
            b = para[5]   # cumulative threshold coefficient b
            k = para[6]   # exponential coefficient

            # Mean SOS over years
            SOS_mean = np.mean(SOS_data)
            alan_mean = np.max(alan_data)

            # Calculate estimated DFS for each year
            for year in range(len(phenology_data)):
                temp01 = temp_data[year, :]
                Cacc = 0

                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", category=RuntimeWarning)
                    # Calculation that may raise warnings
                    Cacc = (Tb - temp01) ** x * (1 - hours_data / Ps) ** y * np.exp(k * ((alan_data[year] - alan_mean) / alan_mean))

                # Apply conditions to zero out Cacc
                Cacc = np.where((temp01 >= Tb) | (hours_data >= Ps) | (np.arange(len(temp01)) < 183), 0, Cacc)
                Sf = np.cumsum(Cacc)

                Ycrit = a + b * ((SOS_data[year] - SOS_mean) / SOS_mean)
                DFS = np.argmax(Sf >= Ycrit) + 1  # +1 to convert from index to day
                if Sf.max() < Ycrit:
                    sim.append(365)
                else:
                    sim.append(DFS)

            sim = np.array(sim)

            rmse = np.sqrt(np.mean((sim - phenology_data) ** 2))
          
            return rmse
        
        def constraint(para, phenology_data, temp_data, hours_data, SOS_data, alan_data):
            a = para[4]
            b = para[5]
           
            # Mean SOS over years
            SOS_mean = np.mean(SOS_data)
            
            return a + b * ((SOS_data[year] - SOS_mean) / SOS_mean)


        # Define lower and upper bounds (photoperiod, temperature threshold, x, y, cumulative threshold parameters)
        # Reference from MATLAB code and Liu Qiang GCB paper
        lb = [8, 0, 0, 0, 0, -1000, -2]
        ub = [24, 50, 2, 2, 30000, 1000, 2]  # Trying max temp 50

        # xopt rmse corresponds to PSO output g and fg
        xopt, rmse, fg_histor = pso(
            objective_function,
            lb,
            ub,
            ieqcons=[constraint], 
            args=(phenology_pixel, temp_copy, hours_pixel, SOS_pixel, alan_pixel),
            swarmsize=50,   # Number of particles
            maxiter=100     # Maximum iterations
        )

        Ps, Tb, x, y, a, b, k = xopt

        # Return 7 parameters and RMSE
        return Ps, Tb, x, y, a, b, k, rmse

    except Exception as e:
        return [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan]


# Main program
# Initialize row and column counts

output_folder = r'H:\DFS_models\result\SIAM_ALAN\SIAM_alan_para'
os.makedirs(output_folder, exist_ok=True)

# Load data

DFS_data = np.load(r'H:\DFS_models\DFS.npy')
DFS_data = DFS_data[:, 1:23]

print(DFS_data.shape)

SOS_data = np.load(r'H:\DFS_models\SOS.npy')
SOS_data = SOS_data[:, 1:23]

print(SOS_data.shape)

hours_data = np.load(r'H:\DFS_models\photoperiod.npy')
hours_data = hours_data[:, 1:366]

print(hours_data.shape)

temp_data = np.load(r'H:\DFS_models\temp.npy')
print(temp_data.shape)

ALAN_data = np.load(r'H:\DFS_models\ALAN.npy')
ALAN_data = ALAN_data[:, 1:23]
print(ALAN_data.shape)

indices = np.arange(452)

results = Parallel(n_jobs=-1)(
            delayed(process_single_pixel)(
                idx, DFS_data, temp_data, hours_data, SOS_data, ALAN_data
            )
            for idx in tqdm(indices)
)

results = np.array(results)
print(results.shape)

np.save(os.path.join(output_folder, 'SIAM_alan_para.npy'), results)


####  Calculate model performance

In [ ]:
import os
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyswarm import pso
import importlib
import pyswarm
importlib.reload(pyswarm)
from joblib import Parallel, delayed
from tqdm import tqdm
import netCDF4 as nc
import warnings
from scipy.stats import pearsonr


def KGE(obs, pre):
    """
    Calculate Kling-Gupta Efficiency (KGE)
    
    Parameters:
        obs: ndarray, observed values
        pre: ndarray, predicted values
        
    Returns:
        kge_value: float, KGE value
    """
    # Calculate Pearson correlation coefficient
    r, _ = pearsonr(obs, pre)
    
    # Calculate mean and standard deviation
    m_obs = np.mean(obs)  # mean of observed values
    m_pre = np.mean(pre)  # mean of predicted values
    std_obs = np.std(obs)  # std of observed values
    std_pre = np.std(pre)  # std of predicted values
    
    # Calculate Kling-Gupta Efficiency
    kge_value = 1 - np.sqrt((r - 1)**2 + (std_pre / std_obs - 1)**2 + (m_pre / m_obs - 1)**2)
    
    return kge_value


def calculate_AIC(observed, simulated, k):
    """
    Calculate AIC (Akaike Information Criterion)
    Reference: the RSE paper on nighttime lights
    
    Parameters:
        observed: ndarray, observed values
        simulated: ndarray, simulated values
        k: int, number of model parameters
        
    Returns:
        aic: float, AIC value
    """
    n = len(observed)
    residual = observed - simulated
    mse = np.mean(residual**2)  # mean squared error
    aic = n * np.log(mse) + 2 * k
    return aic


# Only pixels with average phenology values participate (mask)
def cal_phenology(para_pixel, phenology_pixel, temp_copy, hours_pixel, SOS_pixel, alan_pixel):
    """
    Python implementation of SM.phenology function.
    
    Parameters:
        para_pixel: parameter list [Ps, Tb, x, y, a, b, k]
        phenology_pixel: observed phenology data
        temp_copy: temperature data (years x days)
        hours_pixel: photoperiod data (days)
        SOS_pixel: start of season data
        alan_pixel: ALAN data
        
    Returns:
        rmse, corr, p_value, kge, aic_value
    """
    sim = []

    Ps = para_pixel[0]  # photoperiod
    Tb = para_pixel[1]  # temperature threshold
    x = para_pixel[2]   # parameter x
    y = para_pixel[3]   # parameter y
    a = para_pixel[4]   # cumulative threshold a
    b = para_pixel[5]   # cumulative threshold b
    k = para_pixel[6]   # exponential coefficient
    
    # Mean SOS over years
    SOS_mean = np.mean(SOS_pixel)
    alan_mean = np.max(alan_pixel)
    
    # Calculate estimated DFS for each year
    for year in range(22):
        temp01 = temp_copy[year, :]
        Cacc = 0
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            # Possible warnings from computation
            Cacc = (Tb - temp01) ** x * (1 - hours_pixel / Ps) ** y * np.exp(k * ((alan_pixel[year] - alan_mean) / alan_mean))

        # Zero out Cacc based on conditions
        Cacc = np.where((temp01 >= Tb) | (hours_pixel >= Ps) | (np.arange(len(temp01)) < 183), 0, Cacc)
        Sf = np.cumsum(Cacc)
        Ycrit = a + b * ((SOS_pixel[year] - SOS_mean) / SOS_mean)  # 22-dimensional
        DFS = np.argmax(Sf >= Ycrit) + 1  # +1 to convert from index to day
        if Sf.max() < Ycrit:
            sim.append(365)
        else:
            sim.append(DFS)

    sim = np.array(sim)

    # Calculate RMSE
    rmse = np.sqrt(np.mean((sim - phenology_pixel) ** 2))

    # Calculate Pearson correlation coefficient and p-value
    corr, p_value = pearsonr(sim, phenology_pixel)

    # Calculate KGE
    kge = KGE(phenology_pixel, sim)

    k = 6
    aic_value = calculate_AIC(phenology_pixel, sim, k)

    return rmse, corr, p_value, kge, aic_value


def mad_based_nan(points, threshold=2.5):
    """Use MAD to identify outliers and replace them with NaN"""
    points = np.copy(points)
    median = np.median(points, axis=0)  # calculate median
    diff = np.abs(points - median)  # absolute deviation from median
    mad = np.median(diff)  # calculate MAD
    outlier_mask = diff > (threshold * mad)  # mark outliers beyond threshold
    points[outlier_mask] = np.nan  # replace outliers with NaN
    return points


def process_single_pixel(col_idx, para_row_data, phenology_row_data, temp_row_data, hours_row_data, SOS_row_data, alan_row_data):
    # Process a single pixel

    phenology_pixel = phenology_row_data[col_idx, :]  # get all years data for column col_idx
    SOS_pixel = SOS_row_data[col_idx, :]
    alan_pixel = alan_row_data[col_idx, :]
    para_pixel = para_row_data[col_idx, :]  # get 7 parameters
    temp_pixel = temp_row_data[col_idx, :, :]  # get all years and days data
    para_pixel = para_pixel.squeeze()

    hours_pixel = hours_row_data[col_idx, :].squeeze()
    alan_pixel = alan_pixel.squeeze()
    phenology_pixel = phenology_pixel.squeeze()
    temp_pixel = temp_pixel.squeeze()
    SOS_pixel = SOS_pixel.squeeze()

    # Data cleaning - remove NaNs
    if np.isnan(temp_pixel).all():
        return [np.nan, np.nan, np.nan, np.nan, np.nan]

    temp_copy = np.full_like(temp_pixel, np.nan, dtype=np.float32)

    for year in range(22):  # Corresponds to 2003-2020
        temp_year = temp_pixel[year, :]
        if np.isnan(temp_year).sum() > 0:
            temp_year = pd.Series(temp_year).interpolate(
                method='linear', limit_direction='both'
            ).fillna(method='ffill').fillna(method='bfill').to_numpy()
        temp_copy[year, :] = temp_year

    phenology_pixel = mad_based_nan(phenology_pixel)

    # If values <180 or NaNs exceed 70%, skip this pixel
    if ((phenology_pixel < 180).sum() + np.isnan(phenology_pixel).sum()) > 0.7 * len(phenology_pixel):
        return [np.nan, np.nan, np.nan, np.nan, np.nan]

    # Fill missing values
    if np.isnan(phenology_pixel).sum() > 0:
        phenology_pixel = pd.Series(phenology_pixel).interpolate(
            method='linear', limit_direction='both'
        ).fillna(method='ffill').fillna(method='bfill').to_numpy()

    sim_DFS = cal_phenology(para_pixel, phenology_pixel, temp_copy, hours_pixel, SOS_pixel, alan_pixel)

    return sim_DFS


output_folder = r'H:\DFS_models\result\SIAM_ALAN\SIAM_alan_performance'
os.makedirs(output_folder, exist_ok=True)

# Load data

DFS_data = np.load(r'H:\DFS_models\DFS.npy')
DFS_data = DFS_data[:, 1:23]
print(DFS_data.shape)

SOS_data = np.load(r'H:\DFS_models\SOS.npy')
SOS_data = SOS_data[:, 1:23]
print(SOS_data.shape)

hours_data = np.load(r'H:\DFS_models\photoperiod.npy')
hours_data = hours_data[:, 1:366]
print(hours_data.shape)

temp_data = np.load(r'H:\DFS_models\temp.npy')
print(temp_data.shape)

ALAN_data = np.load(r'H:\DFS_models\ALAN.npy')
ALAN_data = ALAN_data[:, 1:23]
print(ALAN_data.shape)

para_data = np.load(r'H:\DFS_models\result\SIAM_ALAN\SIAM_alan_para\SIAM_alan_para.npy')
print(para_data.shape)

indices = np.arange(452)

results = Parallel(n_jobs=-1)(
    delayed(process_single_pixel)(
        idx, para_data, DFS_data, temp_data, hours_data, SOS_data, ALAN_data
    )
    for idx in tqdm(indices)
)

results = np.array(results)
print(results.shape)

np.save(os.path.join(output_folder, 'SIAM_alan_performance.npy'), results)


In [ ]:
import numpy as np

# Load the original array
CDD_performance = np.load(r"H:\DFS_models\result\SIAM_ALAN\SIAM_alan_performance\SIAM_alan_performance.npy")

# Ensure the second dimension is 5
print("Shape:", CDD_performance.shape)

# Split and save
names = ['rmse', 'corr', 'p_value', 'kge', 'aic_value']
for i, name in enumerate(names):
    np.save(fr"H:\DFS_models\result\SIAM_ALAN\SIAM_alan_performance\{name}.npy", CDD_performance[:, i])


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

# Input and output directories
input_dir = r"H:\DFS_models\result\SIAM_ALAN\SIAM_alan_performance"
output_dir = r"H:\DFS_models\result\SIAM_ALAN\SIAM_alan_performance\distribution"

# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)

# Iterate over all .npy files in the directory
for filename in os.listdir(input_dir):
    if filename.endswith(".npy"):
        file_path = os.path.join(input_dir, filename)
        print("Processing:", filename)
        
        # Load the .npy file
        data = np.load(file_path)
        
        # Remove NaN values
        valid_data = data[~np.isnan(data)]

        # Get the base filename without extension
        base_filename = os.path.splitext(filename)[0]

        # Compute the density distribution
        density = gaussian_kde(valid_data)
        x_vals = np.linspace(valid_data.min(), valid_data.max(), 500)
        y_vals = density(x_vals)

        # Plot the density distribution
        plt.figure(figsize=(8, 6))
        plt.plot(x_vals, y_vals, color='darkblue', lw=2, label="Density")
        plt.fill_between(x_vals, y_vals, color='skyblue', alpha=0.4)
        plt.xlabel(f"{base_filename}", fontsize=14)  # Add filename as x-axis label
        plt.ylabel("Density", fontsize=14)
        plt.legend(fontsize=12)
        plt.grid(alpha=0.3)
        
        # Save the density plot
        density_output_path = os.path.join(output_dir, f"{base_filename}_density.png")
        plt.savefig(density_output_path)
        plt.close()

        print(f"Completed processing: {filename}")

print("All files processed!")


### 7. DMT model

#### Calculate the optimal model parameters

In [ ]:
import os
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyswarm import pso
import importlib
import pyswarm
importlib.reload(pyswarm)
from joblib import Parallel, delayed
from tqdm import tqdm
import netCDF4 as nc
import warnings


def plot_pso_convergence(global_best_fitness, row, col):
    """Plot PSO convergence curve"""
    plt.figure(figsize=(10, 6))
    plt.plot(global_best_fitness, marker='o', color='blue', label="PSO Convergence")
    plt.title(f"PSO Convergence Trend (Row: {row}, Col: {col})")
    plt.xlabel("Iteration")
    plt.ylabel("Global Best Fitness")
    plt.grid()
    plt.legend()
    plt.show()


def mad_based_nan(points, threshold=2.5):
    """Use MAD to identify outliers and replace them with NaN"""
    points = points.copy()  # Remove read-only restriction
    median = np.median(points, axis=0)  # Calculate median
    diff = np.abs(points - median)  # Absolute deviation from median
    mad = np.median(diff)  # Calculate MAD
    outlier_mask = diff > (threshold * mad)  # Flag outliers exceeding threshold
    points[outlier_mask] = np.nan  # Replace outliers with NaN
    return points


def process_single_pixel(col_idx, phenology_row_data, temp_row_data, hours_row_data, SOS_row_data):
    # Process single pixel
    try:
        phenology_pixel = phenology_row_data[col_idx, :]  # Get all years data for the col-th pixel
        SOS_pixel = SOS_row_data[col_idx, :]  # Get all years data for the col-th pixel
        temp_pixel = temp_row_data[col_idx, :, :]  # Get all years and days data for the col-th pixel
        phenology_pixel = phenology_pixel.squeeze()
        SOS_pixel = SOS_pixel.squeeze()
        temp_pixel = temp_pixel.squeeze()
        # print(temp_pixel.shape) #(22,365)

        hours_pixel = hours_row_data[col_idx, :]
        hours_pixel = hours_pixel.squeeze()

        if np.isnan(temp_pixel).all():  # If all temp data is NaN
            return [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan]

        # Data cleaning, remove NaNs
        temp_copy = np.full_like(temp_pixel, np.nan, dtype=np.float32)

        for year in range(22):  # Corresponding to 2001-2022
            temp_year = temp_pixel[year, :]

            if np.isnan(temp_year).sum() > 0:
                temp_year = pd.Series(temp_year).interpolate(
                    method='linear', limit_direction='both'
                ).fillna(method='ffill').fillna(method='bfill').to_numpy()
            temp_copy[year, :] = temp_year

        phenology_pixel = mad_based_nan(phenology_pixel)

        # If the number of values less than 180 plus NaNs is more than 70% of total data, skip
        if ((phenology_pixel < 180).sum() + np.isnan(phenology_pixel).sum()) > 0.7 * len(phenology_pixel):
            return [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan]

        # Fill missing values
        if np.isnan(phenology_pixel).sum() > 0:
            phenology_pixel = pd.Series(phenology_pixel).interpolate(
                method='linear', limit_direction='both'
            ).fillna(method='ffill').fillna(method='bfill').to_numpy()

        def objective_function(para, phenology_data, temp_data, hours_data, SOS_data):
            """
            Python implementation of the SM.phenology function.

            Parameters:
                para: list of parameters [Ps, Tb, x, y, a, b]
                temp_data: temperature data, shape (years, days)
                hours_data: phenology data, shape (days,), same for every year
                phenology_data: observed phenology data

            Returns:
                rmse: calculated root mean squared error
            """
            sim = []

            Ps = para[0]  # Photoperiod
            Tb = para[1]  # Temperature threshold
            x = para[2]   # Parameter x
            y = para[3]   # Parameter y
            a = para[4]   # Accumulation threshold parameter a
            b = para[5]   # Accumulation threshold parameter b

            SOS_mean = np.mean(SOS_data)

            # Calculate DFS estimate for each year
            for year in range(len(phenology_data)):
                temp01 = temp_data[year, :]
                Cacc = 0

                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", category=RuntimeWarning)
                    Cacc = (Tb - temp01) ** x * (1 - hours_data / Ps) ** y

                Cacc = np.where((temp01 >= Tb) | (hours_data >= Ps) | (np.arange(len(temp01)) < 183), 0, Cacc)
                Sf = np.cumsum(Cacc)

                Ycrit = a + b * ((SOS_data[year] - SOS_mean) / SOS_mean)
                DFS = np.argmax(Sf >= Ycrit) + 1  # +1 to convert from index to day
                if Sf.max() < Ycrit:
                    sim.append(365)
                else:
                    sim.append(DFS)

            sim = np.array(sim)

            rmse = np.sqrt(np.mean((sim - phenology_data) ** 2))

            return rmse

        def constraint(para, phenology_data, temp_data, hours_data, SOS_data):
            a = para[4]
            b = para[5]
            SOS_mean = np.mean(SOS_data)
            return a + b * ((SOS_data[year] - SOS_mean) / SOS_mean)

        # Define parameter bounds (photoperiod, temperature threshold, x, y, accumulation thresholds)
        lb = [8, 0, 0, 0, 0, -1000]
        ub = [24, 50, 2, 2, 30000, 1000]  # Try max temperature = 50

        # Run PSO optimizer: xopt is optimal params, rmse is objective function value, fg_histor is convergence history
        xopt, rmse, fg_histor = pso(
            objective_function,
            lb,
            ub,
            ieqcons=[constraint],
            args=(phenology_pixel, temp_copy, hours_pixel, SOS_pixel),
            swarmsize=50,
            maxiter=100
        )

        Ps, Tb, x, y, a, b = xopt

        # Return 7 parameters including rmse
        return Ps, Tb, x, y, a, b, rmse
    except Exception as e:
        print(f"Error processing site: {e}")
        return [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan]


# Main program
# Initialize row and column count

output_folder = r'H:\DFS_models\result\DMT\DMT_para'
os.makedirs(output_folder, exist_ok=True)

# Load data

DFS_data = np.load(r'H:\DFS_models\DFS.npy')
DFS_data = DFS_data[:, 1:23]

print(DFS_data.shape)

SOS_data = np.load(r'H:\DFS_models\TSS.npy')
SOS_data = SOS_data[:, 1:23]

print(SOS_data.shape)

hours_data = np.load(r'H:\DFS_models\photoperiod.npy')
hours_data = hours_data[:, 1:366]

print(hours_data.shape)

temp_data = np.load(r'H:\DFS_models\temp.npy')
print(temp_data.shape)

indices = np.arange(452)

results = Parallel(n_jobs=-1)(
    delayed(process_single_pixel)(
        idx, DFS_data, temp_data, hours_data, SOS_data
    )
    for idx in tqdm(indices)
)

results = np.array(results)
print(results.shape)

np.save(os.path.join(output_folder, 'DMT_para.npy'), results)


#### Calculate model performance

In [ ]:
import os
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyswarm import pso
import importlib
import pyswarm
importlib.reload(pyswarm)
from joblib import Parallel, delayed
from tqdm import tqdm
import netCDF4 as nc
import warnings
from scipy.stats import pearsonr


def KGE(obs, pre):
    """
    Calculate Kling-Gupta Efficiency (KGE)
    
    Parameters:
        obs: ndarray, observed values
        pre: ndarray, predicted values
        
    Returns:
        kge_value: float, KGE value
    """
    # Calculate Pearson correlation coefficient
    r, _ = pearsonr(obs, pre)
    
    # Calculate mean and standard deviation
    m_obs = np.mean(obs)  # mean of observed values
    m_pre = np.mean(pre)  # mean of predicted values
    std_obs = np.std(obs)  # std of observed values
    std_pre = np.std(pre)  # std of predicted values
    
    # Calculate Kling-Gupta Efficiency
    kge_value = 1 - np.sqrt((r - 1)**2 + (std_pre / std_obs - 1)**2 + (m_pre / m_obs - 1)**2)
    
    return kge_value


def calculate_AIC(observed, simulated, k):
    """
    Calculate AIC (Akaike Information Criterion)
    Reference: RSE from the nighttime lights study
    
    Parameters:
        observed: ndarray, observed values
        simulated: ndarray, simulated values
        
    Returns:
        aic: float, AIC value
    """
    n = len(observed)
    residual = observed - simulated
    mse = np.mean(residual**2)  # mean squared error
    aic = n * np.log(mse) + 2 * k
    return aic



# Only pixels with valid phenology averages participate in calculation (mask)
def cal_phenology(para_pixel, phenology_pixel, temp_copy, hours_pixel, SOS_pixel):
    """
    Python implementation of SM.phenology function.
    
    Parameters:
        para_pixel: parameter list [Ps, Tb, x, y, a, b]
        phenology_pixel: observed phenology data
        temp_copy: temperature data, shape (years, days)
        hours_pixel: photoperiod data, shape (days,)
        SOS_pixel: start of season data
    
    Returns:
        rmse, corr, p_value, kge, aic_value
    """
    sim = []

    Ps = para_pixel[0]  # photoperiod
    Tb = para_pixel[1]  # temperature threshold

    x = para_pixel[2]  # parameter x
    y = para_pixel[3]  # parameter y

    a = para_pixel[4]
    b = para_pixel[5]

    # Mean SOS over multiple years
    SOS_mean = np.mean(SOS_pixel)

    # Calculate estimated DFS for each year
    for year in range(22):
        temp01 = temp_copy[year, :]
        Cacc = 0

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            # Calculate cumulative accumulation with possible runtime warnings ignored
            Cacc = (Tb - temp01) ** x * (1 - hours_pixel / Ps) ** y

        # Set accumulation to zero based on conditions
        Cacc = np.where((temp01 >= Tb) | (hours_pixel >= Ps) | (np.arange(len(temp01)) < 183), 0, Cacc)
        Sf = np.cumsum(Cacc)
        Ycrit = a + b * ((SOS_pixel[year] - SOS_mean) / SOS_mean)  # 22-dimensional vector
        DFS = np.argmax(Sf >= Ycrit) + 1  # +1 to convert from index to day
        if Sf.max() < Ycrit:
            sim.append(365)
        else:
            sim.append(DFS)

    # Calculate RMSE
    rmse = np.sqrt(np.mean((sim - phenology_pixel) ** 2))

    # Calculate Pearson correlation coefficient and p-value
    corr, p_value = pearsonr(sim, phenology_pixel)

    # Calculate KGE
    kge = KGE(phenology_pixel, sim)

    # Calculate AIC
    k = 5  # number of model parameters
    aic_value = calculate_AIC(phenology_pixel, sim, k)

    return rmse, corr, p_value, kge, aic_value



def mad_based_nan(points, threshold=2.5):
    """Use MAD to identify outliers and replace them with NaN"""
    points = np.copy(points)
    median = np.median(points, axis=0)  # calculate median
    diff = np.abs(points - median)  # absolute deviation from median
    mad = np.median(diff)  # calculate MAD
    outlier_mask = diff > (threshold * mad)  # mark outliers beyond threshold
    points[outlier_mask] = np.nan  # replace outliers with NaN
    return points


def process_single_pixel(col_idx, para_row_data, phenology_row_data, temp_row_data, hours_row_data, SOS_row_data):
    # Process single pixel

    phenology_pixel = phenology_row_data[col_idx, :]  # get all years' data of column col_idx
    SOS_pixel = SOS_row_data[col_idx, :]
    para_pixel = para_row_data[col_idx, :]  # get 6 parameters
    temp_pixel = temp_row_data[col_idx, :, :]  # get all years and all days data
    para_pixel = para_pixel.squeeze()

    hours_pixel = hours_row_data[col_idx, :]
    hours_pixel = hours_pixel.squeeze()

    phenology_pixel = phenology_pixel.squeeze()
    temp_pixel = temp_pixel.squeeze()
    para_pixel = para_pixel.squeeze()
    SOS_pixel = SOS_pixel.squeeze()
    # print(temp_pixel.shape) #(22,365)

    # Data cleaning: remove nan values
    if np.isnan(temp_pixel).all():
        return [np.nan, np.nan, np.nan, np.nan, np.nan]

    temp_copy = np.full_like(temp_pixel, np.nan, dtype=np.float32)

    for year in range(22):  # corresponding to 2003-2020
        temp_year = temp_pixel[year, :]

        if np.isnan(temp_year).sum() > 0:
            temp_year = pd.Series(temp_year).interpolate(
                method='linear', limit_direction='both'
            ).fillna(method='ffill').fillna(method='bfill').to_numpy()
        temp_copy[year, :] = temp_year

    # Remove outliers
    phenology_pixel = mad_based_nan(phenology_pixel)

    # If values < 180 or NaNs exceed 70% of total data, exclude pixel
    if ((phenology_pixel < 180).sum() + np.isnan(phenology_pixel).sum()) > 0.7 * len(phenology_pixel):
        return [np.nan, np.nan, np.nan, np.nan, np.nan]

    # Fill missing values
    if np.isnan(phenology_pixel).sum() > 0:
        phenology_pixel = pd.Series(phenology_pixel).interpolate(
            method='linear', limit_direction='both'
        ).fillna(method='ffill').fillna(method='bfill').to_numpy()

    sim_DFS = cal_phenology(para_pixel, phenology_pixel, temp_copy, hours_pixel, SOS_pixel)

    return sim_DFS



output_folder = r'H:\DFS_models\result\DMT\DMT_performance'
os.makedirs(output_folder, exist_ok=True)

# Load data

DFS_data = np.load(r'H:\DFS_models\DFS.npy')
DFS_data = DFS_data[:, 1:23]

print(DFS_data.shape)

SOS_data = np.load(r'H:\DFS_models\TSS.npy')
SOS_data = SOS_data[:, 1:23]

print(SOS_data.shape)

hours_data = np.load(r'H:\DFS_models\photoperiod.npy')
hours_data = hours_data[:, 1:366]

print(hours_data.shape)

temp_data = np.load(r'H:\DFS_models\temp.npy')
print(temp_data.shape)

para_data = np.load(r'H:\DFS_models\result\DMT\DMT_para\DMT_para.npy')
print(para_data.shape)

indices = np.arange(452)

results = Parallel(n_jobs=-1)(
    delayed(process_single_pixel)(
        idx, para_data, DFS_data, temp_data, hours_data, SOS_data
    )
    for idx in tqdm(indices)
)

results = np.array(results)
print(results.shape)

np.save(os.path.join(output_folder, 'DMT_performance.npy'), results)


In [ ]:
import numpy as np

# Load the original array
CDD_performance = np.load(r"H:\DFS_models\result\DMT\DMT_performance\DMT_performance.npy")

print("Shape:", CDD_performance.shape)

# Split and save
names = ['rmse', 'corr', 'p_value', 'kge', 'aic_value']
for i, name in enumerate(names):
    np.save(fr"H:\DFS_models\result\DMT\DMT_performance\{name}.npy", CDD_performance[:, i])


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

# Input and output directories
input_dir = r"H:\DFS_models\result\DMT\DMT_performance"
output_dir = r"H:\DFS_models\result\DMT\DMT_performance\distribution"

# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)

# Iterate over all .npy files in the input directory
for filename in os.listdir(input_dir):
    if filename.endswith(".npy"):
        file_path = os.path.join(input_dir, filename)
        print("Processing:", filename)
        
        # Load the .npy file
        data = np.load(file_path)
        
        # Remove NaN values
        valid_data = data[~np.isnan(data)]
        
        # Get the base filename without extension
        base_filename = os.path.splitext(filename)[0]
        
        # Calculate the density distribution using Gaussian KDE
        density = gaussian_kde(valid_data)
        x_vals = np.linspace(valid_data.min(), valid_data.max(), 500)
        y_vals = density(x_vals)
        
        # Plot the density distribution
        plt.figure(figsize=(8, 6))
        plt.plot(x_vals, y_vals, color='darkblue', lw=2, label="Density")
        plt.fill_between(x_vals, y_vals, color='skyblue', alpha=0.4)
        plt.xlabel(f"{base_filename}", fontsize=14)  # Add filename as x-axis label
        plt.ylabel("Density", fontsize=14)
        plt.legend(fontsize=12)
        plt.grid(alpha=0.3)
        
        # Save the density plot
        density_output_path = os.path.join(output_dir, f"{base_filename}_density.png")
        plt.savefig(density_output_path)
        plt.close()
        
        print(f"Finished processing: {filename}")

print("All files have been processed!")


### 8. DMT_ALAN model

#### Calculate the optimal model parameters

In [ ]:
import os
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyswarm import pso
import importlib
import pyswarm
importlib.reload(pyswarm)
from joblib import Parallel, delayed
from tqdm import tqdm
import netCDF4 as nc
import warnings


def plot_pso_convergence(global_best_fitness, row, col):
    """Plot PSO convergence curve"""
    plt.figure(figsize=(10, 6))
    plt.plot(global_best_fitness, marker='o', color='blue', label="PSO Convergence")
    plt.title(f"PSO Convergence Trend (Row: {row}, Col: {col})")
    plt.xlabel("Iteration")
    plt.ylabel("Global Best Fitness")
    plt.grid()
    plt.legend()
    plt.show()


def mad_based_nan(points, threshold=2.5):
    """Detect outliers using MAD and replace them with NaN"""
    points = points.copy()  # Remove read-only restriction
    median = np.median(points, axis=0)  # Calculate median
    diff = np.abs(points - median)  # Calculate absolute deviation from median
    mad = np.median(diff)  # Calculate MAD
    outlier_mask = diff > (threshold * mad)  # Mark values exceeding threshold as outliers
    points[outlier_mask] = np.nan  # Replace outliers with NaN
    return points


def process_single_pixel(col_idx, phenology_row_data, temp_row_data, hours_row_data, SOS_row_data, alan_row_data):
    """Process a single pixel"""
    try:
        phenology_pixel = phenology_row_data[col_idx, :]  # Get all years data for the column
        SOS_pixel = SOS_row_data[col_idx, :]              # Get all years data for the column
        alan_pixel = alan_row_data[col_idx, :]            # Get all years data for the column
        temp_pixel = temp_row_data[col_idx, :, :]         # Get all years and days data
        phenology_pixel = phenology_pixel.squeeze()
        SOS_pixel = SOS_pixel.squeeze()
        temp_pixel = temp_pixel.squeeze()

        hours_pixel = hours_row_data[col_idx, :]
        hours_pixel = hours_pixel.squeeze()

        if np.isnan(temp_pixel).all():
            return [np.nan] * 8

        # Data cleaning: fill NaNs in temperature data
        temp_copy = np.full_like(temp_pixel, np.nan, dtype=np.float32)

        for year in range(22):  # Corresponds to years 2001-2022
            temp_year = temp_pixel[year, :]
            if np.isnan(temp_year).sum() > 0:
                temp_year = pd.Series(temp_year).interpolate(
                    method='linear', limit_direction='both'
                ).fillna(method='ffill').fillna(method='bfill').to_numpy()
            temp_copy[year, :] = temp_year

        # Remove outliers from phenology data
        phenology_pixel = mad_based_nan(phenology_pixel)

        # If more than 30% of phenology data are less than 180 or NaN, skip
        if ((phenology_pixel < 180).sum() + np.isnan(phenology_pixel).sum()) > 0.7 * len(phenology_pixel):
            return [np.nan] * 8

        # Fill missing values in phenology data
        if np.isnan(phenology_pixel).sum() > 0:
            phenology_pixel = pd.Series(phenology_pixel).interpolate(
                method='linear', limit_direction='both'
            ).fillna(method='ffill').fillna(method='bfill').to_numpy()


        def objective_function(para, phenology_data, temp_data, hours_data, SOS_data, alan_data):
            """
            Python implementation of SM.phenology function.

            Parameters:
                para: parameter list [Ps, Tb, x, y, a, b, k]
                temp_data: temperature data, shape (years, days)
                hours_data: photoperiod data, shape (days,), same for every year
                phenology_data: observed phenology data

            Returns:
                rmse: root mean square error
            """
            sim = []

            Ps = para[0]  # Photoperiod
            Tb = para[1]  # Temperature threshold
            x = para[2]   # Parameter x
            y = para[3]   # Parameter y
            a = para[4]   # Accumulation threshold intercept
            b = para[5]   # Accumulation threshold slope
            k = para[6]   # ALAN effect coefficient

            SOS_mean = np.mean(SOS_data)
            alan_mean = np.nanmax(alan_data)

            for year in range(len(phenology_data)):
                temp01 = temp_data[year, :]
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", category=RuntimeWarning)
                    Cacc = (Tb - temp01) ** x * (1 - hours_data / Ps) ** y * np.exp(k * ((alan_data[year] - alan_mean) / alan_mean))

                Cacc = np.where((temp01 >= Tb) | (hours_data >= Ps) | (np.arange(len(temp01)) < 183), 0, Cacc)
                Sf = np.cumsum(Cacc)
                Ycrit = a + b * ((SOS_data[year] - SOS_mean) / SOS_mean)
                DFS = np.argmax(Sf >= Ycrit) + 1  # +1 to convert from index to day
                if Sf.max() < Ycrit:
                    sim.append(365)
                else:
                    sim.append(DFS)

            sim = np.array(sim)
            rmse = np.sqrt(np.mean((sim - phenology_data) ** 2))
            return rmse

        def constraint(para, phenology_data, temp_data, hours_data, SOS_data, alan_data):
            a = para[4]
            b = para[5]
            SOS_mean = np.mean(SOS_data)
            return a + b * ((SOS_data[year] - SOS_mean) / SOS_mean)

        # Define parameter bounds (photoperiod, temperature threshold, x, y, accumulation threshold, ALAN coefficient)
        lb = [8, 0, 0, 0, 0, -1000, -2]
        ub = [24, 50, 2, 2, 30000, 1000, 2]

        # Run PSO to optimize parameters
        xopt, rmse, fg_history = pso(
            objective_function,
            lb,
            ub,
            ieqcons=[constraint],
            args=(phenology_pixel, temp_copy, hours_pixel, SOS_pixel, alan_pixel),
            swarmsize=50,
            maxiter=100
        )

        Ps, Tb, x, y, a, b, k = xopt

        return Ps, Tb, x, y, a, b, k, rmse
    except Exception as e:
        print(f"Error processing pixel: {e}")
        return [np.nan] * 8


# Main program
output_folder = r'H:\DFS_models\result\DMT_ALAN\DMT_alan_para'
os.makedirs(output_folder, exist_ok=True)

# Load data
DFS_data = np.load(r'H:\DFS_models\DFS.npy')
DFS_data = DFS_data[:, 1:23]
print(DFS_data.shape)

SOS_data = np.load(r'H:\DFS_models\TSS.npy')
SOS_data = SOS_data[:, 1:23]
print(SOS_data.shape)

hours_data = np.load(r'H:\DFS_models\photoperiod.npy')
hours_data = hours_data[:, 1:366]
print(hours_data.shape)

temp_data = np.load(r'H:\DFS_models\temp.npy')
print(temp_data.shape)

ALAN_data = np.load(r'H:\DFS_models\ALAN.npy')
ALAN_data = ALAN_data[:, 1:23]
print(ALAN_data.shape)

indices = np.arange(452)

results = Parallel(n_jobs=-1)(
    delayed(process_single_pixel)(
        idx, DFS_data, temp_data, hours_data, SOS_data, ALAN_data
    )
    for idx in tqdm(indices)
)

results = np.array(results)
print(results.shape)

np.save(os.path.join(output_folder, 'DMT_alan_para.npy'), results)


In [ ]:
import numpy as np

# File path
file_path = r"H:\NightLightPhenology\GlobalNightLightPhenology\08_AutumnPhenologyModel\data_new\urban\result\DMT_ALAN\DMT_alan_para\DMT_alan_para.npy"
# Load the npy file
data = np.load(file_path)

# Extract the 7th column (index 6)
col7 = data[:, 6]

# Print the first few rows to check
print(col7)


#### Calculate the simulated DFS values

In [ ]:
import os
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyswarm import pso
import importlib
import pyswarm
importlib.reload(pyswarm)
from joblib import Parallel, delayed
from tqdm import tqdm
import netCDF4 as nc
import warnings
from scipy.stats import pearsonr


# Only pixels with valid average phenology values participate in the calculation (masking)
def cal_phenology(para_pixel, temp_copy, hours_pixel, SOS_pixel, alan_pixel):
    """
    Python implementation of the SM.phenology function.

    Parameters:
        para_pixel: parameter list [Ps, Tb, x, y, a, b, k]
        temp_copy: temperature data, shape (years, days)
        hours_pixel: photoperiod data, shape (days), same every year
        SOS_pixel: start of season phenology data
        alan_pixel: ALAN (nighttime light) data

    Returns:
        sim: simulated DFS (end of season) array for each year
    """
    sim = []

    Ps = para_pixel[0]  # photoperiod threshold
    Tb = para_pixel[1]  # temperature threshold
    x = para_pixel[2]   # parameter x
    y = para_pixel[3]   # parameter y
    a = para_pixel[4]   # parameter a
    b = para_pixel[5]   # parameter b
    k = para_pixel[6]   # ALAN influence parameter

    # Multi-year average SOS
    SOS_mean = np.mean(SOS_pixel)
    # Use max ALAN as reference
    alan_mean = np.max(alan_pixel)

    # Calculate estimated DFS for each year
    for year in range(22):
        temp01 = temp_copy[year, :]
        Cacc = 0

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            # Calculation potentially generating warnings
            Cacc = (Tb - temp01) ** x * (1 - hours_pixel / Ps) ** y * np.exp(k * ((alan_pixel[year] - alan_mean) / alan_mean))

        # Set accumulation to zero under given conditions
        Cacc = np.where((temp01 >= Tb) | (hours_pixel >= Ps) | (np.arange(len(temp01)) < 183), 0, Cacc)
        Sf = np.cumsum(Cacc)
        Ycrit = a + b * ((SOS_pixel[year] - SOS_mean) / SOS_mean)  # 22-length array
        DFS = np.argmax(Sf >= Ycrit) + 1  # +1 converts index to day of year

        if Sf.max() < Ycrit:
            sim.append(365)
        else:
            sim.append(DFS)

    sim = np.array(sim)
    return sim


def mad_based_nan(points, threshold=2.5):
    """Identify outliers using Median Absolute Deviation (MAD) and replace them with NaN."""
    points = np.copy(points)
    median = np.median(points, axis=0)  # median
    diff = np.abs(points - median)  # absolute deviation from median
    mad = np.median(diff)  # MAD
    outlier_mask = diff > (threshold * mad)  # mask for outliers
    points[outlier_mask] = np.nan  # replace outliers with NaN
    return points


def process_single_pixel(col_idx, para_row_data, temp_row_data, hours_row_data, SOS_row_data, alan_row_data):
    """Process a single pixel at column index col_idx."""
    SOS_pixel = SOS_row_data[col_idx, :]
    alan_pixel = alan_row_data[col_idx, :]
    para_pixel = para_row_data[col_idx, :]  # get 7 parameters
    temp_pixel = temp_row_data[col_idx, :, :]  # all years, all days temperature data

    para_pixel = para_pixel.squeeze()
    hours_pixel = hours_row_data[col_idx, :].squeeze()
    alan_pixel = alan_pixel.squeeze()
    temp_pixel = temp_pixel.squeeze()
    SOS_pixel = SOS_pixel.squeeze()

    temp_copy = np.full_like(temp_pixel, np.nan, dtype=np.float32)

    # Interpolate missing temperature data for each year
    for year in range(22):
        temp_year = temp_pixel[year, :]
        if np.isnan(temp_year).sum() > 0:
            temp_year = pd.Series(temp_year).interpolate(method='linear', limit_direction='both').fillna(method='ffill').fillna(method='bfill').to_numpy()
        temp_copy[year, :] = temp_year

    # Calculate simulated DFS using phenology model
    sim_DFS = cal_phenology(para_pixel, temp_copy, hours_pixel, SOS_pixel, alan_pixel)

    return sim_DFS


# Main program

output_folder = r'H:\DFS_models\result\DMT_ALAN\DMT_alan_DFS'
os.makedirs(output_folder, exist_ok=True)

# Load data

DFS_data = np.load(r'H:\DFS_models\DFS.npy')
DFS_data = DFS_data[:, 1:23]
print(DFS_data.shape)

SOS_data = np.load(r'H:\DFS_models\TSS.npy')
SOS_data = SOS_data[:, 1:23]
print(SOS_data.shape)

hours_data = np.load(r'H:\DFS_models\photoperiod.npy')
hours_data = hours_data[:, 1:366]
print(hours_data.shape)

temp_data = np.load(r'H:\DFS_models\temp.npy')
print(temp_data.shape)

ALAN_data = np.load(r'H:\DFS_models\ALAN.npy')
ALAN_data = ALAN_data[:, 1:23]
print(ALAN_data.shape)

para_data = np.load(r'H:\DFS_models\result\DMT_ALAN\DMT_alan_para\DMT_alan_para.npy')
print(para_data.shape)

indices = np.arange(452)

results = Parallel(n_jobs=-1)(
    delayed(process_single_pixel)(
        idx, para_data, temp_data, hours_data, SOS_data, ALAN_data
    ) for idx in tqdm(indices)
)

results = np.array(results)
print(results.shape)

np.save(os.path.join(output_folder, 'DMT_alan_DFS.npy'), results)

# Save results to Excel
df = pd.DataFrame(results)
df.to_excel(os.path.join(output_folder, 'DMT_alan_DFS.xlsx'), index=False)


#### Calculate model performance

In [ ]:
import os
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyswarm import pso
import importlib
import pyswarm
importlib.reload(pyswarm)
from joblib import Parallel, delayed
from tqdm import tqdm
import netCDF4 as nc
import warnings
from scipy.stats import pearsonr


def KGE(obs, pre):
    """
    Calculate Kling-Gupta Efficiency (KGE)

    Parameters:
        obs: ndarray, observed values
        pre: ndarray, predicted values

    Returns:
        kge_value: float, KGE value
    """
    r, _ = pearsonr(obs, pre)
    m_obs = np.mean(obs)
    m_pre = np.mean(pre)
    std_obs = np.std(obs)
    std_pre = np.std(pre)
    kge_value = 1 - np.sqrt((r - 1)**2 + (std_pre / std_obs - 1)**2 + (m_pre / m_obs - 1)**2)
    return kge_value


def calculate_AIC(observed, simulated, k):
    """
    Calculate AIC (Akaike Information Criterion)
    Reference: The RSE paper on night lights.

    Parameters:
        observed: ndarray, observed values
        simulated: ndarray, predicted values
        k: int, number of model parameters

    Returns:
        aic: float, AIC value
    """
    n = len(observed)
    residual = observed - simulated
    mse = np.mean(residual**2)
    aic = n * np.log(mse) + 2 * k
    return aic


def cal_phenology(para_pixel, phenology_pixel, temp_copy, hours_pixel, SOS_pixel, alan_pixel):
    """
    Python implementation of the SM.phenology function.

    Parameters:
        para_pixel: model parameters for a pixel
        phenology_pixel: observed phenology values (DFS)
        temp_copy: temperature data (years x days)
        hours_pixel: daylength values (same for each year)
        SOS_pixel: start-of-season data
        alan_pixel: ALAN values for each year

    Returns:
        Various evaluation metrics (RMSE, correlation, p-value, KGE, AIC)
    """
    sim = []

    Ps = para_pixel[0]
    Tb = para_pixel[1]
    x = para_pixel[2]
    y = para_pixel[3]
    a = para_pixel[4]
    b = para_pixel[5]
    k = para_pixel[6]

    SOS_mean = np.mean(SOS_pixel)
    alan_mean = np.max(alan_pixel)

    for year in range(22):
        temp01 = temp_copy[year, :]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            Cacc = (Tb - temp01) ** x * (1 - hours_pixel / Ps) ** y * np.exp(k * ((alan_pixel[year] - alan_mean) / alan_mean))
        Cacc = np.where((temp01 >= Tb) | (hours_pixel >= Ps) | (np.arange(len(temp01)) < 183), 0, Cacc)
        Sf = np.cumsum(Cacc)
        Ycrit = a + b * ((SOS_pixel[year] - SOS_mean) / SOS_mean)
        DFS = np.argmax(Sf >= Ycrit) + 1
        if Sf.max() < Ycrit:
            sim.append(365)
        else:
            sim.append(DFS)

    sim = np.array(sim)

    rmse = np.sqrt(np.mean((sim - phenology_pixel) ** 2))
    corr, p_value = pearsonr(sim, phenology_pixel)
    kge = KGE(phenology_pixel, sim)
    k = 6
    aic_value = calculate_AIC(phenology_pixel, sim, k)

    return rmse, corr, p_value, kge, aic_value


def mad_based_nan(points, threshold=2.5):
    """ Identify outliers using MAD and replace them with NaN """
    points = np.copy(points)
    median = np.median(points, axis=0)
    diff = np.abs(points - median)
    mad = np.median(diff)
    outlier_mask = diff > (threshold * mad)
    points[outlier_mask] = np.nan
    return points


def process_single_pixel(col_idx, para_row_data, phenology_row_data, temp_row_data, hours_row_data, SOS_row_data, alan_row_data):
    """ Process a single pixel's time series """
    phenology_pixel = phenology_row_data[col_idx, :]
    SOS_pixel = SOS_row_data[col_idx, :]
    alan_pixel = alan_row_data[col_idx, :]
    para_pixel = para_row_data[col_idx, :]
    temp_pixel = temp_row_data[col_idx, :, :]
    hours_pixel = hours_row_data[col_idx, :]

    para_pixel = para_pixel.squeeze()
    hours_pixel = hours_pixel.squeeze()
    alan_pixel = alan_pixel.squeeze()
    phenology_pixel = phenology_pixel.squeeze()
    temp_pixel = temp_pixel.squeeze()
    SOS_pixel = SOS_pixel.squeeze()

    if np.isnan(temp_pixel).all():
        return [np.nan, np.nan, np.nan, np.nan, np.nan]

    temp_copy = np.full_like(temp_pixel, np.nan, dtype=np.float32)

    for year in range(22):
        temp_year = temp_pixel[year, :]
        if np.isnan(temp_year).sum() > 0:
            temp_year = pd.Series(temp_year).interpolate(method='linear', limit_direction='both').fillna(method='ffill').fillna(method='bfill').to_numpy()
        temp_copy[year, :] = temp_year

    phenology_pixel = mad_based_nan(phenology_pixel)

    if ((phenology_pixel < 180).sum() + np.isnan(phenology_pixel).sum()) > 0.7 * len(phenology_pixel):
        return [np.nan, np.nan, np.nan, np.nan, np.nan]

    if np.isnan(phenology_pixel).sum() > 0:
        phenology_pixel = pd.Series(phenology_pixel).interpolate(method='linear', limit_direction='both').fillna(method='ffill').fillna(method='bfill').to_numpy()

    sim_DFS = cal_phenology(para_pixel, phenology_pixel, temp_copy, hours_pixel, SOS_pixel, alan_pixel)
    return sim_DFS


# Output folder
output_folder = r'H:\DFS_models\result\DMT_ALAN\DMT_alan_performance'
os.makedirs(output_folder, exist_ok=True)

# Load data
DFS_data = np.load(r'H:\DFS_models\DFS.npy')[:, 1:23]
print(DFS_data.shape)

SOS_data = np.load(r'H:\DFS_models\TSS.npy')[:, 1:23]
print(SOS_data.shape)

hours_data = np.load(r'H:\DFS_models\photoperiod.npy')[:, 1:366]
print(hours_data.shape)

temp_data = np.load(r'H:\DFS_models\temp.npy')
print(temp_data.shape)

ALAN_data = np.load(r'H:\DFS_models\ALAN.npy')[:, 1:23]
print(ALAN_data.shape)

para_data = np.load(r'H:\DFS_models\result\DMT_ALAN\DMT_alan_para\DMT_alan_para.npy')
print(para_data.shape)

# Indices for processing
indices = np.arange(452)

# Parallel processing
results = Parallel(n_jobs=-1)(
    delayed(process_single_pixel)(
        idx, para_data, DFS_data, temp_data, hours_data, SOS_data, ALAN_data
    )
    for idx in tqdm(indices)
)

results = np.array(results)
print(results.shape)
np.save(os.path.join(output_folder, 'DMT_alan_performance.npy'), results)


In [ ]:
import numpy as np

# Load the original array
CDD_performance = np.load(r"H:\DFS_models\result\DMT_ALAN\DMT_alan_performance\DMT_alan_performance.npy")

# Ensure the second dimension has 5 columns
print("Shape:", CDD_performance.shape)

# Split and save each column separately
names = ['rmse', 'corr', 'p_value', 'kge', 'aic_value']
for i, name in enumerate(names):
    np.save(fr"H:\DFS_models\result\DMT_ALAN\DMT_alan_performance\{name}.npy", CDD_performance[:, i])



In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

# Input and output directories
input_dir = r"H:\DFS_models\result\DMT_ALAN\DMT_alan_performance"
output_dir = r"H:\DFS_models\result\DMT_ALAN\DMT_alan_performance\distribution"

# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)

# Iterate through all .npy files in the directory
for filename in os.listdir(input_dir):
    if filename.endswith(".npy"):
        file_path = os.path.join(input_dir, filename)
        print("Processing:", filename)

        # Load the .npy file
        data = np.load(file_path)

        # Remove NaN values
        valid_data = data[~np.isnan(data)]

        # Get the base filename (without extension)
        base_filename = os.path.splitext(filename)[0]

        # Compute density distribution
        density = gaussian_kde(valid_data)
        x_vals = np.linspace(valid_data.min(), valid_data.max(), 500)
        y_vals = density(x_vals)

        # Plot the density distribution
        plt.figure(figsize=(8, 6))
        plt.plot(x_vals, y_vals, color='darkblue', lw=2, label="Density")
        plt.fill_between(x_vals, y_vals, color='skyblue', alpha=0.4)
        plt.xlabel(f"{base_filename}", fontsize=14)  # Add the filename to the x-axis label
        plt.ylabel("Density", fontsize=14)
        plt.legend(fontsize=12)
        plt.grid(alpha=0.3)

        # Save the density distribution plot
        density_output_path = os.path.join(output_dir, f"{base_filename}_density.png")
        plt.savefig(density_output_path)
        plt.close()

        print(f"Finished processing: {filename}")

print("All files processed!")

